In [21]:
!pip install -U openai pandas numpy tqdm python-dotenv pyarrow

In [91]:
from __future__ import annotations

import os
import re
import json
import uuid
import pickle
import hashlib
import datetime as dt
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key="")


# -----------------------------
# Judge API settings
# -----------------------------
JUDGE_PROVIDER = "openai"
JUDGE_MODEL = "gpt-5.4"

# GPT-5.4 rejected temperature in the failed Batch job, so DO NOT send temperature.
# Error was: Unsupported parameter: 'temperature' is not supported with this model.
JUDGE_REASONING_EFFORT = "low"
JUDGE_TEXT_VERBOSITY = "low"

# Give hidden reasoning enough room. Visible output should still be one integer.
JUDGE_MAX_OUTPUT_TOKENS = 256

# Do not send optional cache params in the rerun. Keep the request body minimal.
PROMPT_CACHE_RETENTION = None
PROMPT_CACHE_KEY = None

# Unique rerun tag so we recollect the entire batch and do not collide with failed request IDs.
JUDGE_RERUN_TAG = "notemp_reasonlow_maxtok256_fullrerun"

# -----------------------------
# Correct analysis input paths
# -----------------------------
FINAL_ANALYSIS_DIR = Path("final_analysis")
FINAL_INPUT_DIR = FINAL_ANALYSIS_DIR / "input_data"

# Original base/dyad/triad experiment
EXPERIMENT_DATA_PATH = FINAL_INPUT_DIR / "experiment_data.pkl"

# Follow-up methods: simplestrat5 and g2_css_static3
FOLLOWUP_STAGED_ROOT = FINAL_INPUT_DIR / "followup_new_baselines"
FOLLOWUP_OUTPUTS_PATH = (
    FOLLOWUP_STAGED_ROOT
    / "r2_outputs"
    / "new_baseline_outputs_all_providers.parquet"
)

assert EXPERIMENT_DATA_PATH.exists(), f"Missing original experiment data: {EXPERIMENT_DATA_PATH}"
assert FOLLOWUP_OUTPUTS_PATH.exists(), f"Missing follow-up outputs: {FOLLOWUP_OUTPUTS_PATH}"

print("Original experiment data:")
print(EXPERIMENT_DATA_PATH)
print("\nFollow-up outputs:")
print(FOLLOWUP_OUTPUTS_PATH)

# -----------------------------
# Design constants
# -----------------------------
PROVIDER_ORDER = ["openai", "anthropic", "gemini"]

PROVIDER_MODEL_CANONICAL = {
    "openai": "gpt-5.4",
    "anthropic": "claude-sonnet-4-6",
    "gemini": "gemini-2.5-pro",
}

EXPECTED_SLOGAN_TASKS = [
    "slogan_smartphone",
    "slogan_soda",
    "slogan_blood_donation",
]

TASK_FAMILY = {
    "slogan_smartphone": "slogan",
    "slogan_soda": "slogan",
    "slogan_blood_donation": "slogan",
}

TASK_LABEL = {
    "slogan_smartphone": "Smartphone slogan",
    "slogan_soda": "Soda slogan",
    "slogan_blood_donation": "Blood donation slogan",
}

EXPECTED_STRATEGIES = ["vanilla", "diverge"]

# Analysis-internal method names -> judge/manuscript names
METHOD_RENAME_FOR_JUDGE = {
    "one_shot": "indep",
    "self_rewrite_e0": "self",
    "self_plus_1_peer": "peer1",
    "self_plus_2_peers": "peer2",
    "simplestrat5": "strat",
    "g2_css_static3": "repr",
}

EXPECTED_METHODS_ANALYSIS = list(METHOD_RENAME_FOR_JUDGE.keys())
EXPECTED_METHODS_JUDGE = ["indep", "self", "peer1", "peer2", "strat", "repr"]

EXPECTED_N_EVALUATED_PER_CELL = 150

# -----------------------------
# Output folders
# -----------------------------
RUN_ID = dt.datetime.now().strftime("%Y%m%d_%H%M%S") + "__" + uuid.uuid4().hex[:8]
EXPERIMENT_ID = "slogan_llm_judge_gpt54_creativity"

DATA_ROOT = (
    Path("ai_data")
    / "deflect_creativity"
    / "judging"
    / f"judge_{JUDGE_MODEL}"
    / EXPERIMENT_ID
    / f"run_{RUN_ID}"
)

DIRS = {
    "metadata": DATA_ROOT / "00_metadata",
    "inputs": DATA_ROOT / "01_inputs",
    "plans": DATA_ROOT / "02_plans",
    "batch_inputs": DATA_ROOT / "03_batch_inputs",
    "manifests": DATA_ROOT / "04_manifests",
    "raw_outputs": DATA_ROOT / "05_raw_outputs",
    "raw_errors": DATA_ROOT / "06_raw_errors",
    "parsed": DATA_ROOT / "07_parsed",
    "compiled": DATA_ROOT / "08_compiled",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("\nRun directory:")
print(DATA_ROOT)

Original experiment data:
final_analysis/input_data/experiment_data.pkl

Follow-up outputs:
final_analysis/input_data/followup_new_baselines/r2_outputs/new_baseline_outputs_all_providers.parquet

Run directory:
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf


In [92]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path: Path) -> list[dict]:
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


def stable_hash(text: str, n: int = 24) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:n]


def clean_model_text(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    text = str(text).strip()
    if len(text) >= 2 and ((text[0] == text[-1] == '"') or (text[0] == text[-1] == "'")):
        text = text[1:-1].strip()
    return text


def first_existing_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise RuntimeError(
        f"None of these candidate columns found: {candidates}\n"
        f"Available columns:\n{df.columns.tolist()}"
    )


def json_safe(obj: Any) -> Any:
    if obj is None:
        return None
    if isinstance(obj, (str, int, bool)):
        return obj
    if isinstance(obj, float):
        return None if (np.isnan(obj) or np.isinf(obj)) else obj
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        val = float(obj)
        return None if (np.isnan(val) or np.isinf(val)) else val
    if isinstance(obj, np.ndarray):
        return [json_safe(x) for x in obj.tolist()]
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(x) for x in obj]
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return str(obj)


def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    raise ValueError(f"Unsupported table format: {path}")


def resolve_existing_path(candidates: list[Path], label: str) -> Path:
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p

    print(f"Could not find {label}. Tried:")
    for p in candidates:
        print(" -", p)
    raise FileNotFoundError(label)


def normalize_strategy(x):
    x = str(x).strip().lower()
    if x in {"neutral", "vanilla", "n"}:
        return "vanilla"
    if x in {"diverge", "divergent", "d"}:
        return "diverge"
    return x


def original_method_from_round_condition(row):
    cond = str(row["condition"]).strip().lower()
    rnd = int(float(row["round"]))

    if cond == "base" and rnd == 1:
        return "one_shot"
    if cond == "base" and rnd == 2:
        return "self_rewrite_e0"
    if cond == "dyad" and rnd == 2:
        return "self_plus_1_peer"
    if cond == "triad" and rnd == 2:
        return "self_plus_2_peers"

    # dyad/triad round 1 are seed rows, not evaluated pools
    return None

In [93]:
@dataclass
class ExperimentData:
    long_df: Optional[pd.DataFrame] = None
    embeddings: Optional[np.ndarray] = None


@dataclass
class ProviderExperimentData:
    long_df: Optional[pd.DataFrame] = None
    embeddings: Optional[np.ndarray] = None
    provider: Optional[str] = None
    provider_label: Optional[str] = None
    model: Optional[str] = None


class GenericPicklePlaceholder:
    def __init__(self, *args, **kwargs):
        self.__dict__.update(kwargs)

    def __setstate__(self, state):
        if isinstance(state, dict):
            self.__dict__.update(state)
        else:
            self.__dict__["state"] = state


class CompatibleUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == "__main__":
            if name == "ExperimentData":
                return ExperimentData
            if name == "ProviderExperimentData":
                return ProviderExperimentData
            return GenericPicklePlaceholder
        return super().find_class(module, name)


def load_experiment_data(path: Path) -> ExperimentData:
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Run the final analysis setup first, "
            "or update EXPERIMENT_DATA_PATH."
        )

    with open(path, "rb") as f:
        obj = CompatibleUnpickler(f).load()

    if hasattr(obj, "long_df") and obj.long_df is not None:
        return obj

    candidate_attrs = getattr(obj, "__dict__", {})
    provider_objects = []

    for attr_value in candidate_attrs.values():
        if isinstance(attr_value, dict):
            provider_objects.extend(
                [v for v in attr_value.values() if hasattr(v, "long_df") and v.long_df is not None]
            )
        elif isinstance(attr_value, (list, tuple)):
            provider_objects.extend(
                [v for v in attr_value if hasattr(v, "long_df") and v.long_df is not None]
            )

    if provider_objects:
        long_parts = [p.long_df for p in provider_objects]
        emb_parts = [
            np.asarray(p.embeddings)
            for p in provider_objects
            if hasattr(p, "embeddings") and p.embeddings is not None
        ]
        return ExperimentData(
            long_df=pd.concat(long_parts, ignore_index=True, sort=False),
            embeddings=np.vstack(emb_parts) if emb_parts else None,
        )

    raise RuntimeError("Loaded pickle object does not contain usable long_df.")

In [94]:
JUDGE_SYSTEM_PROMPT = """
You are an expert evaluator of short advertising and public-campaign slogans.

Your task is to score one candidate slogan for slogan creativity.

You will be given:
1. The slogan-writing task context.
2. One candidate slogan.

Do not infer or consider which model, method, prompt condition, or experimental treatment produced the slogan. That information is intentionally hidden.

Before answering, silently consider the slogan's task fit, clarity, fluency, slogan-likeness, memorability, persuasive appeal, and creative expression. Do not output your reasoning.

Score the slogan on an absolute 1-5 scale for slogan creativity.

For this task, slogan creativity means the slogan is both:
1. appropriate and effective for the specified product or campaign; and
2. creatively expressed as a concise slogan.

A high-scoring slogan should fit the task, be fluent and clear, feel slogan-like, and have memorable, distinctive, clever, or persuasive phrasing.

Do not reward novelty by itself. A strange, confusing, awkward, off-task, or unpersuasive slogan should receive a low score even if it is unusual. A fluent but generic slogan should receive a moderate score unless it is especially effective.

Use this scale:
1 = Poor: off-task, confusing, awkward, inappropriate, or not usable as a slogan.
2 = Weak: relevant but generic, flat, vague, cliched, or weakly persuasive.
3 = Adequate: relevant, fluent, and usable as a slogan, but not especially memorable or creative.
4 = Good: relevant, fluent, slogan-like, and clearly memorable, distinctive, clever, or persuasive.
5 = Excellent: highly effective and creatively strong; polished, memorable, persuasive, and immediately usable.

Do not penalize the slogan for exceeding the requested word limit. Judge the slogan's creative effectiveness as written.

Return only one integer from 1 to 5. Do not return JSON, explanation, punctuation, or any other text.
""".strip()


SLOGAN_TASK_CONTEXT = {
    "slogan_smartphone": """
You are part of the marketing team at a tech company preparing to launch a new smartphone.
The candidate is intended to be a marketing slogan for this brand-new smartphone.
The slogan should be written in English.
You may assume any detail about the smartphone.
""".strip(),

    "slogan_soda": """
You are part of the marketing team at a beverage company preparing to launch a new soda.
The candidate is intended to be a marketing slogan for this brand-new soda.
The slogan should be written in English.
You may assume any detail about the soda.
""".strip(),

    "slogan_blood_donation": """
You are part of the communications team at a nonprofit organization preparing a campaign to encourage blood donation.
The candidate is intended to be a campaign slogan for this blood donation campaign.
The slogan should be written in English.
You may assume any detail about the campaign.
""".strip(),
}


def build_judge_user_prompt(task_id: str, slogan: str) -> str:
    if task_id not in SLOGAN_TASK_CONTEXT:
        raise ValueError(f"Unknown slogan task_id: {task_id}")

    return f"""
Slogan-writing task context:
{SLOGAN_TASK_CONTEXT[task_id]}

Candidate slogan:
{slogan}

Silently apply the rubric, then score this slogan for slogan creativity.

Return only one integer from 1 to 5.
""".strip()

In [95]:
run_config = {
    "experiment_id": EXPERIMENT_ID,
    "judge_provider": JUDGE_PROVIDER,
    "judge_model": JUDGE_MODEL,
    "judge_temperature_sent": False,
    "judge_temperature_note": "temperature omitted because GPT-5.4 rejected temperature in Batch/Responses with unsupported_parameter.",
    "judge_reasoning_effort": JUDGE_REASONING_EFFORT,
    "judge_text_verbosity": JUDGE_TEXT_VERBOSITY,
    "judge_max_output_tokens": JUDGE_MAX_OUTPUT_TOKENS,
    "prompt_cache_retention": PROMPT_CACHE_RETENTION,
    "prompt_cache_key": PROMPT_CACHE_KEY,
    "judge_rerun_tag": JUDGE_RERUN_TAG,
    "experiment_data_path": str(EXPERIMENT_DATA_PATH),
    "followup_outputs_path": str(FOLLOWUP_OUTPUTS_PATH),
    "score_name": "slogan_creativity_1to5",
    "one_slogan_per_request": True,
    "length_compliance_penalized": False,
    "created_at_utc": now_iso(),
    "data_root": str(DATA_ROOT),
    "run_id": RUN_ID,
}

config_path = DIRS["metadata"] / f"slogan_judge_config__{JUDGE_RERUN_TAG}__{RUN_ID}.json"
write_json(config_path, run_config)
config_path

PosixPath('ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/00_metadata/slogan_judge_config__notemp_reasonlow_maxtok256_fullrerun__20260525_030656__bf5164cf.json')

In [96]:
# Cell 6 — Load original experiment + follow-up strat/repr data

# -----------------------------
# A. Load original base/dyad/triad data
# -----------------------------
experiment_data = load_experiment_data(EXPERIMENT_DATA_PATH)
old_df = experiment_data.long_df.copy()
old_df["_old_row_pos"] = np.arange(len(old_df))

TEXT_COL_OLD = first_existing_col(
    old_df,
    ["text", "response_text", "output_text", "model_output", "clean_text", "idea", "response"],
)

old = old_df.copy()

required_old_cols = ["provider", "task_id", "strategy", "condition", "round"]
missing_old_cols = [c for c in required_old_cols if c not in old.columns]
if missing_old_cols:
    raise RuntimeError(f"Original data missing required columns: {missing_old_cols}")

if "model" not in old.columns:
    old["model"] = old["provider"].map(PROVIDER_MODEL_CANONICAL).fillna(old["provider"])

old["provider"] = old["provider"].astype(str)
old["model"] = old["model"].astype(str)
old["task_id"] = old["task_id"].astype(str)
old["task_family"] = old["task_id"].map(TASK_FAMILY)
old["task_label"] = old["task_id"].map(TASK_LABEL)
old["strategy"] = old["strategy"].map(normalize_strategy)
old["condition"] = old["condition"].astype(str)
old["round"] = pd.to_numeric(old["round"], errors="coerce").astype("Int64")
old["method_analysis"] = old.apply(original_method_from_round_condition, axis=1)

# Keep only evaluated original slogan pools:
# base/r1 -> one_shot
# base/r2 -> self_rewrite_e0
# dyad/r2 -> self_plus_1_peer
# triad/r2 -> self_plus_2_peers
old = old[
    old["provider"].isin(PROVIDER_ORDER)
    & old["task_id"].isin(EXPECTED_SLOGAN_TASKS)
    & old["task_family"].eq("slogan")
    & old["strategy"].isin(EXPECTED_STRATEGIES)
    & old["method_analysis"].isin(EXPECTED_METHODS_ANALYSIS)
].copy()

old["source_stage"] = "original"
old["text_clean"] = old[TEXT_COL_OLD].map(clean_model_text)

if "request_key" not in old.columns:
    old["request_key"] = np.nan

old["slot_num"] = (
    old.groupby(["provider", "task_id", "method_analysis", "strategy"], observed=True)
    .cumcount()
    .add(1)
)

old["slot_id"] = old["slot_num"].map(lambda x: f"slot_{x:03d}")

old["row_uid"] = old[
    [
        "source_stage",
        "provider",
        "model",
        "round",
        "task_id",
        "method_analysis",
        "strategy",
        "condition",
        "slot_id",
    ]
].astype(str).agg("|".join, axis=1).map(lambda x: stable_hash(x, 32))

print("Prepared original evaluated slogan rows:", old.shape)
display(
    old.groupby(["provider", "task_id", "method_analysis", "strategy"], observed=True)
    .size()
    .reset_index(name="n")
    .sort_values(["provider", "task_id", "method_analysis", "strategy"])
)

# Sanity check: original should be 10,800 rows
expected_old_n = (
    len(PROVIDER_ORDER)
    * len(EXPECTED_SLOGAN_TASKS)
    * 4
    * len(EXPECTED_STRATEGIES)
    * EXPECTED_N_EVALUATED_PER_CELL
)
if len(old) != expected_old_n:
    raise RuntimeError(f"Expected {expected_old_n:,} original evaluated slogan rows, found {len(old):,}.")


# -----------------------------
# B. Load follow-up strat/repr data
# -----------------------------
followup_raw = read_table(FOLLOWUP_OUTPUTS_PATH).copy()
followup = followup_raw.copy()

print("\nLoaded follow-up raw:", followup.shape)
print("Follow-up path:", FOLLOWUP_OUTPUTS_PATH)

TEXT_COL_FOLLOWUP = first_existing_col(
    followup,
    ["text", "response", "output_text", "model_output", "clean_text", "idea"],
)

required_followup_cols = ["provider", "task_id", "strategy"]
missing_followup_cols = [c for c in required_followup_cols if c not in followup.columns]
if missing_followup_cols:
    raise RuntimeError(f"Follow-up outputs missing required columns: {missing_followup_cols}")

if "model" not in followup.columns:
    followup["model"] = followup["provider"].map(PROVIDER_MODEL_CANONICAL).fillna(followup["provider"])

# The full analysis notebook expects a method column here.
# If method is absent, fall back to condition.
if "method" not in followup.columns:
    if "condition" in followup.columns:
        followup["method"] = followup["condition"]
    else:
        raise RuntimeError("Follow-up outputs missing both method and condition columns.")

followup["provider"] = followup["provider"].astype(str)
followup["model"] = followup["model"].astype(str)
followup["task_id"] = followup["task_id"].astype(str)
followup["task_family"] = followup["task_id"].map(TASK_FAMILY)
followup["task_label"] = followup["task_id"].map(TASK_LABEL)
followup["strategy"] = followup["strategy"].map(normalize_strategy)
followup["method_analysis"] = followup["method"].astype(str)
followup["round"] = pd.to_numeric(followup["round"], errors="coerce").fillna(1).astype(int) if "round" in followup.columns else 1
followup["condition"] = followup["condition"] if "condition" in followup.columns else followup["method_analysis"]
followup["source_stage"] = "followup"
followup["text_clean"] = followup[TEXT_COL_FOLLOWUP].map(clean_model_text)

# Keep only follow-up evaluated slogan pools:
# simplestrat5 -> strat
# g2_css_static3 -> repr
followup = followup[
    followup["provider"].isin(PROVIDER_ORDER)
    & followup["task_id"].isin(EXPECTED_SLOGAN_TASKS)
    & followup["task_family"].eq("slogan")
    & followup["strategy"].isin(EXPECTED_STRATEGIES)
    & followup["method_analysis"].isin(["simplestrat5", "g2_css_static3"])
].copy()

# Slot handling
if "slot_id" not in followup.columns:
    followup["slot_num"] = (
        followup.groupby(["provider", "task_id", "method_analysis", "strategy"], observed=True)
        .cumcount()
        .add(1)
    )
    followup["slot_id"] = followup["slot_num"].map(lambda x: f"slot_{x:03d}")
else:
    followup["slot_id"] = followup["slot_id"].astype(str)
    extracted_slot = followup["slot_id"].str.extract(r"(\d+)")[0]
    followup["slot_num"] = pd.to_numeric(extracted_slot, errors="coerce")

    missing_slot = followup["slot_num"].isna()
    if missing_slot.any():
        followup.loc[missing_slot, "slot_num"] = (
            followup.loc[missing_slot]
            .groupby(["provider", "task_id", "method_analysis", "strategy"], observed=True)
            .cumcount()
            .add(1)
        )

    followup["slot_num"] = followup["slot_num"].astype(int)

if "request_key" not in followup.columns:
    followup["request_key"] = np.nan

followup["row_uid"] = followup[
    [
        "source_stage",
        "provider",
        "model",
        "task_id",
        "method_analysis",
        "strategy",
        "slot_id",
    ]
].astype(str).agg("|".join, axis=1).map(lambda x: stable_hash(x, 32))

print("\nPrepared follow-up evaluated slogan rows:", followup.shape)
display(
    followup.groupby(["provider", "task_id", "method_analysis", "strategy"], observed=True)
    .size()
    .reset_index(name="n")
    .sort_values(["provider", "task_id", "method_analysis", "strategy"])
)

# Sanity check: follow-up should be 5,400 rows
expected_followup_n = (
    len(PROVIDER_ORDER)
    * len(EXPECTED_SLOGAN_TASKS)
    * 2
    * len(EXPECTED_STRATEGIES)
    * EXPECTED_N_EVALUATED_PER_CELL
)
if len(followup) != expected_followup_n:
    raise RuntimeError(f"Expected {expected_followup_n:,} follow-up evaluated slogan rows, found {len(followup):,}.")


# -----------------------------
# C. Combine original + follow-up evaluated slogan pools
# -----------------------------
COMMON_COLS = sorted(set(old.columns).union(followup.columns))

slogan_outputs_all = pd.concat(
    [
        old.reindex(columns=COMMON_COLS),
        followup.reindex(columns=COMMON_COLS),
    ],
    ignore_index=True,
    sort=False,
)

slogan_outputs_all["is_success"] = slogan_outputs_all["text_clean"].astype(str).str.strip().ne("")
slogan_outputs_all = slogan_outputs_all[slogan_outputs_all["is_success"]].copy().reset_index(drop=True)

slogan_outputs_all["method"] = slogan_outputs_all["method_analysis"].map(METHOD_RENAME_FOR_JUDGE)
slogan_outputs_all["slogan_text"] = slogan_outputs_all["text_clean"]
slogan_outputs_all["_source_row_pos"] = np.arange(len(slogan_outputs_all))
slogan_outputs_all["_provider_model_key"] = (
    slogan_outputs_all["provider"].astype(str)
    + " :: "
    + slogan_outputs_all["model"].astype(str)
)

if slogan_outputs_all["method"].isna().any():
    bad_methods = slogan_outputs_all[slogan_outputs_all["method"].isna()]["method_analysis"].unique()
    raise RuntimeError(f"Unmapped method_analysis values: {bad_methods}")

if slogan_outputs_all["row_uid"].duplicated().any():
    dupes = slogan_outputs_all[slogan_outputs_all["row_uid"].duplicated(keep=False)]
    display(
        dupes[
            [
                "row_uid",
                "source_stage",
                "provider",
                "task_id",
                "method_analysis",
                "strategy",
                "request_key",
            ]
        ].head(100)
    )
    raise RuntimeError("Duplicate row_uid values.")

slogan_df = slogan_outputs_all.copy()

print("\nCombined evaluated slogan rows:", slogan_df.shape)
display(
    slogan_df
    .groupby(["source_stage", "provider", "task_id", "method", "strategy"], observed=True)
    .size()
    .reset_index(name="n")
    .sort_values(["source_stage", "provider", "task_id", "method", "strategy"])
)

# Final expected total: 16,200
expected_total_n = (
    len(PROVIDER_ORDER)
    * len(EXPECTED_SLOGAN_TASKS)
    * len(EXPECTED_METHODS_JUDGE)
    * len(EXPECTED_STRATEGIES)
    * EXPECTED_N_EVALUATED_PER_CELL
)

if len(slogan_df) != expected_total_n:
    raise RuntimeError(f"Expected {expected_total_n:,} total evaluated slogan rows, found {len(slogan_df):,}.")

print(f"\nOK: loaded {len(slogan_df):,} evaluated slogans.")

Prepared original evaluated slogan rows: (10800, 58)


,provider,task_id,method_analysis,strategy,n
0,anthropic,slogan_blood_donation,one_shot,vanilla,150
1,anthropic,slogan_blood_donation,one_shot,diverge,150
2,anthropic,slogan_blood_donation,self_plus_1_peer,vanilla,150
3,anthropic,slogan_blood_donation,self_plus_1_peer,diverge,150
4,anthropic,slogan_blood_donation,self_plus_2_peers,vanilla,150
...,...,...,...,...,...
67,openai,slogan_soda,self_plus_1_peer,diverge,150
68,openai,slogan_soda,self_plus_2_peers,vanilla,150
69,openai,slogan_soda,self_plus_2_peers,diverge,150
70,openai,slogan_soda,self_rewrite_e0,vanilla,150



Loaded follow-up raw: (21600, 56)
Follow-up path: final_analysis/input_data/followup_new_baselines/r2_outputs/new_baseline_outputs_all_providers.parquet

Prepared follow-up evaluated slogan rows: (5400, 60)


,provider,task_id,method_analysis,strategy,n
0,anthropic,slogan_blood_donation,g2_css_static3,diverge,150
1,anthropic,slogan_blood_donation,g2_css_static3,vanilla,150
2,anthropic,slogan_blood_donation,simplestrat5,diverge,150
3,anthropic,slogan_blood_donation,simplestrat5,vanilla,150
4,anthropic,slogan_smartphone,g2_css_static3,diverge,150
5,anthropic,slogan_smartphone,g2_css_static3,vanilla,150
6,anthropic,slogan_smartphone,simplestrat5,diverge,150
7,anthropic,slogan_smartphone,simplestrat5,vanilla,150
8,anthropic,slogan_soda,g2_css_static3,diverge,150
9,anthropic,slogan_soda,g2_css_static3,vanilla,150



Combined evaluated slogan rows: (16200, 85)


/var/folders/rj/l30_wb7d3w7_tbx4gbz6lzzh0000gn/T/ipykernel_74487/3328020809.py:213: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  slogan_outputs_all = pd.concat(


,source_stage,provider,task_id,method,strategy,n
0,followup,anthropic,slogan_blood_donation,repr,diverge,150
1,followup,anthropic,slogan_blood_donation,repr,vanilla,150
2,followup,anthropic,slogan_blood_donation,strat,diverge,150
3,followup,anthropic,slogan_blood_donation,strat,vanilla,150
4,followup,anthropic,slogan_smartphone,repr,diverge,150
...,...,...,...,...,...,...
103,original,openai,slogan_soda,peer1,vanilla,150
104,original,openai,slogan_soda,peer2,diverge,150
105,original,openai,slogan_soda,peer2,vanilla,150
106,original,openai,slogan_soda,self,diverge,150



OK: loaded 16,200 evaluated slogans.


In [97]:
provider_model_keys = sorted(slogan_df["_provider_model_key"].unique())

print("Detected provider/model cells:")
for x in provider_model_keys:
    print(" -", x)

if len(provider_model_keys) != 3:
    raise RuntimeError(f"Expected 3 provider/model cells, found {len(provider_model_keys)}")

evaluated_counts = (
    slogan_df
    .groupby(["_provider_model_key", "task_id", "method", "strategy"], observed=True)
    .size()
    .reset_index(name="n")
)

display(
    evaluated_counts.sort_values(["_provider_model_key", "task_id", "method", "strategy"])
)

expected_cells = pd.MultiIndex.from_product(
    [
        provider_model_keys,
        EXPECTED_SLOGAN_TASKS,
        EXPECTED_METHODS_JUDGE,
        EXPECTED_STRATEGIES,
    ],
    names=["_provider_model_key", "task_id", "method", "strategy"],
).to_frame(index=False)

evaluated_audit = expected_cells.merge(
    evaluated_counts,
    on=["_provider_model_key", "task_id", "method", "strategy"],
    how="left",
)

evaluated_audit["n"] = evaluated_audit["n"].fillna(0).astype(int)
evaluated_audit["expected_n"] = EXPECTED_N_EVALUATED_PER_CELL
evaluated_audit["ok"] = evaluated_audit["n"].eq(evaluated_audit["expected_n"])

bad = evaluated_audit[~evaluated_audit["ok"]].copy()

if len(bad):
    display(bad)
    raise RuntimeError("Evaluated slogan coverage check failed.")

expected_total = (
    len(provider_model_keys)
    * len(EXPECTED_SLOGAN_TASKS)
    * len(EXPECTED_METHODS_JUDGE)
    * len(EXPECTED_STRATEGIES)
    * EXPECTED_N_EVALUATED_PER_CELL
)

observed_total = len(slogan_df)

print(f"Expected evaluated slogan rows to judge: {expected_total:,}")
print(f"Observed evaluated slogan rows to judge: {observed_total:,}")

if observed_total != expected_total:
    raise RuntimeError(f"Expected {expected_total:,}, observed {observed_total:,}")

validated_input_csv_path = DIRS["inputs"] / f"slogans_to_judge_VALIDATED_EVALUATED_ONLY__{RUN_ID}.csv"
validated_input_pkl_path = DIRS["inputs"] / f"slogans_to_judge_VALIDATED_EVALUATED_ONLY__{RUN_ID}.pkl"
coverage_audit_csv_path = DIRS["inputs"] / f"slogan_evaluated_coverage_audit__{RUN_ID}.csv"

slogan_df.to_csv(validated_input_csv_path, index=False)
slogan_df.to_pickle(validated_input_pkl_path)
evaluated_audit.to_csv(coverage_audit_csv_path, index=False)

print("Coverage check passed.")
print(validated_input_csv_path)
print(validated_input_pkl_path)
print(coverage_audit_csv_path)

Detected provider/model cells:
 - anthropic :: claude-sonnet-4-6
 - gemini :: gemini-2.5-pro
 - openai :: gpt-5.4


,_provider_model_key,task_id,method,strategy,n
0,anthropic :: claude-sonnet-4-6,slogan_blood_donation,indep,diverge,150
1,anthropic :: claude-sonnet-4-6,slogan_blood_donation,indep,vanilla,150
2,anthropic :: claude-sonnet-4-6,slogan_blood_donation,peer1,diverge,150
3,anthropic :: claude-sonnet-4-6,slogan_blood_donation,peer1,vanilla,150
4,anthropic :: claude-sonnet-4-6,slogan_blood_donation,peer2,diverge,150
...,...,...,...,...,...
103,openai :: gpt-5.4,slogan_soda,repr,vanilla,150
104,openai :: gpt-5.4,slogan_soda,self,diverge,150
105,openai :: gpt-5.4,slogan_soda,self,vanilla,150
106,openai :: gpt-5.4,slogan_soda,strat,diverge,150


Expected evaluated slogan rows to judge: 16,200
Observed evaluated slogan rows to judge: 16,200
Coverage check passed.
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/01_inputs/slogans_to_judge_VALIDATED_EVALUATED_ONLY__20260525_030656__bf5164cf.csv
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/01_inputs/slogans_to_judge_VALIDATED_EVALUATED_ONLY__20260525_030656__bf5164cf.pkl
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/01_inputs/slogan_evaluated_coverage_audit__20260525_030656__bf5164cf.csv


In [98]:
def build_judge_plan(slogan_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    metadata_keep_cols = [
        "_source_row_pos",
        "row_uid",
        "source_stage",
        "provider",
        "model",
        "task_id",
        "task_label",
        "task_family",
        "method",
        "method_analysis",
        "strategy",
        "condition",
        "round",
        "group_id",
        "agent_id",
        "agent_index",
        "slot_id",
        "slot_num",
        "request_key",
        "_provider_model_key",
    ]
    metadata_keep_cols = [c for c in metadata_keep_cols if c in slogan_df.columns]

    for _, row in slogan_df.iterrows():
        basis = {
            "rerun_tag": JUDGE_RERUN_TAG,
            "row_uid": str(row["row_uid"]),
            "source_row_pos": int(row["_source_row_pos"]),
            "provider": str(row["provider"]),
            "model": str(row["model"]),
            "task_id": str(row["task_id"]),
            "method": str(row["method"]),
            "strategy": str(row["strategy"]),
            "slogan_text_hash": stable_hash(row["slogan_text"], 16),
        }

        judge_request_key = "sloganjudge__" + stable_hash(json.dumps(basis, sort_keys=True), 32)

        out = {
            "judge_request_key": judge_request_key,
            "judge_provider": JUDGE_PROVIDER,
            "judge_model": JUDGE_MODEL,
            "judge_temperature_sent": False,
            "judge_reasoning_effort": JUDGE_REASONING_EFFORT,
            "judge_text_verbosity": JUDGE_TEXT_VERBOSITY,
            "judge_max_output_tokens": JUDGE_MAX_OUTPUT_TOKENS,
            "judge_rerun_tag": JUDGE_RERUN_TAG,
            "score_name": "slogan_creativity_1to5",
            "system_instructions": JUDGE_SYSTEM_PROMPT,
            "user_prompt": build_judge_user_prompt(str(row["task_id"]), str(row["slogan_text"])),
            "slogan_text": row["slogan_text"],
            "created_at_utc": now_iso(),
        }

        for c in metadata_keep_cols:
            out[c] = row[c]

        rows.append(out)

    plan = pd.DataFrame(rows)

    if plan["judge_request_key"].duplicated().any():
        dupes = plan[plan["judge_request_key"].duplicated(keep=False)].sort_values("judge_request_key")
        display(dupes.head(20))
        raise RuntimeError("Duplicate judge_request_key detected.")

    if len(plan) != len(slogan_df):
        raise RuntimeError("Judge plan length does not match slogan_df length.")

    expected_total = (
        len(PROVIDER_ORDER)
        * len(EXPECTED_SLOGAN_TASKS)
        * len(EXPECTED_METHODS_JUDGE)
        * len(EXPECTED_STRATEGIES)
        * EXPECTED_N_EVALUATED_PER_CELL
    )

    if len(plan) != expected_total:
        raise RuntimeError(f"Expected {expected_total:,} judge requests, got {len(plan):,}.")

    return plan


judge_plan_df = build_judge_plan(slogan_df)

plan_csv_path = DIRS["plans"] / f"slogan_judge_plan__{JUDGE_RERUN_TAG}__{RUN_ID}.csv"
plan_pkl_path = DIRS["plans"] / f"slogan_judge_plan__{JUDGE_RERUN_TAG}__{RUN_ID}.pkl"

judge_plan_df.to_csv(plan_csv_path, index=False)
judge_plan_df.to_pickle(plan_pkl_path)

print("Judge requests:", len(judge_plan_df))
print(plan_csv_path)
print(plan_pkl_path)

display(judge_plan_df.head())

Judge requests: 16200
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/02_plans/slogan_judge_plan__notemp_reasonlow_maxtok256_fullrerun__20260525_030656__bf5164cf.csv
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/02_plans/slogan_judge_plan__notemp_reasonlow_maxtok256_fullrerun__20260525_030656__bf5164cf.pkl


,judge_request_key,judge_provider,judge_model,judge_temperature_sent,judge_reasoning_effort,judge_text_verbosity,judge_max_output_tokens,judge_rerun_tag,score_name,system_instructions,...,strategy,condition,round,group_id,agent_id,agent_index,slot_id,slot_num,request_key,_provider_model_key
0,sloganjudge__f967b307f0861efc91a8fdd20cac9bc8,openai,gpt-5.4,False,low,low,256,notemp_reasonlow_maxtok256_fullrerun,slogan_creativity_1to5,You are an expert evaluator of short advertisi...,...,vanilla,base,1,base_001,base_001__a1,1.0,slot_001,1,r1__ae37e6a4459629b922a98b61,openai :: gpt-5.4
1,sloganjudge__054c9c0ef01e16a3f62b82d80ca37a72,openai,gpt-5.4,False,low,low,256,notemp_reasonlow_maxtok256_fullrerun,slogan_creativity_1to5,You are an expert evaluator of short advertisi...,...,vanilla,base,2,base_001,base_001__a1,1.0,slot_001,1,r2__4523e53313d039e7622c16a1,openai :: gpt-5.4
2,sloganjudge__695c58007bf270423de45dde2bc1e3e4,openai,gpt-5.4,False,low,low,256,notemp_reasonlow_maxtok256_fullrerun,slogan_creativity_1to5,You are an expert evaluator of short advertisi...,...,vanilla,base,1,base_002,base_002__a1,1.0,slot_002,2,r1__8cb17f81c524a85271342c9d,openai :: gpt-5.4
3,sloganjudge__addb67461a913d296b4e8f11ee345efa,openai,gpt-5.4,False,low,low,256,notemp_reasonlow_maxtok256_fullrerun,slogan_creativity_1to5,You are an expert evaluator of short advertisi...,...,vanilla,base,2,base_002,base_002__a1,1.0,slot_002,2,r2__d6ecab12e1f888e7b2aa16df,openai :: gpt-5.4
4,sloganjudge__43d47f528bccb01d150d6136edee7f0c,openai,gpt-5.4,False,low,low,256,notemp_reasonlow_maxtok256_fullrerun,slogan_creativity_1to5,You are an expert evaluator of short advertisi...,...,vanilla,base,1,base_003,base_003__a1,1.0,slot_003,3,r1__57bc8ea5d603182f310443f0,openai :: gpt-5.4


In [99]:
def make_openai_responses_batch_jsonl(plan_df: pd.DataFrame, batch_jsonl_path: Path) -> Path:
    if batch_jsonl_path.exists():
        raise FileExistsError(f"Refusing to overwrite: {batch_jsonl_path}")

    batch_jsonl_path.parent.mkdir(parents=True, exist_ok=True)

    with open(batch_jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            body = {
                "model": JUDGE_MODEL,
                "instructions": row["system_instructions"],
                "input": row["user_prompt"],
                "max_output_tokens": int(row["judge_max_output_tokens"]),
                "reasoning": {"effort": str(row["judge_reasoning_effort"])},
                "text": {"verbosity": str(row["judge_text_verbosity"])},
            }

            # DO NOT include temperature. GPT-5.4 rejected it:
            # Unsupported parameter: 'temperature' is not supported with this model.
            #
            # DO NOT include prompt_cache_retention/prompt_cache_key in this rerun.
            # Keep the request body minimal to avoid another unsupported-parameter failure.

            request = {
                "custom_id": row["judge_request_key"],
                "method": "POST",
                "url": "/v1/responses",
                "body": body,
            }

            f.write(json.dumps(json_safe(request), ensure_ascii=False) + "\n")

    print(f"Wrote batch input: {batch_jsonl_path}")
    print(f"Requests: {len(plan_df):,}")

    # Quick local validation: confirm temperature is absent from every line.
    with open(batch_jsonl_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            rec = json.loads(line)
            body = rec.get("body", {})
            if "temperature" in body:
                raise RuntimeError(f"temperature unexpectedly found in request line {i}")
            if "prompt_cache_retention" in body or "prompt_cache_key" in body:
                raise RuntimeError(f"cache parameter unexpectedly found in request line {i}")

    print("Local JSONL validation passed: no temperature/cache params.")
    return batch_jsonl_path


batch_jsonl_path = DIRS["batch_inputs"] / f"slogan_judge_batch_input__{JUDGE_RERUN_TAG}__{RUN_ID}.jsonl"

make_openai_responses_batch_jsonl(
    plan_df=judge_plan_df,
    batch_jsonl_path=batch_jsonl_path,
)

Wrote batch input: ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/03_batch_inputs/slogan_judge_batch_input__notemp_reasonlow_maxtok256_fullrerun__20260525_030656__bf5164cf.jsonl
Requests: 16,200
Local JSONL validation passed: no temperature/cache params.


PosixPath('ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/03_batch_inputs/slogan_judge_batch_input__notemp_reasonlow_maxtok256_fullrerun__20260525_030656__bf5164cf.jsonl')

In [100]:
def submit_openai_batch(
    batch_jsonl_path: Path,
    plan_path: Path,
    manifest_dir: Path,
) -> dict:
    batch_input_file = client.files.create(
        file=open(batch_jsonl_path, "rb"),
        purpose="batch",
    )

    batch = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "project": "deflect_creativity",
            "experiment_id": EXPERIMENT_ID,
            "stage": "slogan_llm_judge",
            "judge_provider": JUDGE_PROVIDER,
            "judge_model": JUDGE_MODEL,
            "judge_rerun_tag": JUDGE_RERUN_TAG,
            "temperature_sent": "false",
            "run_id": RUN_ID,
            "local_input_file": str(batch_jsonl_path),
            "local_plan_file": str(plan_path),
        },
    )

    batch_info = {
        "run_id": RUN_ID,
        "experiment_id": EXPERIMENT_ID,
        "stage": "slogan_llm_judge",
        "judge_provider": JUDGE_PROVIDER,
        "judge_model": JUDGE_MODEL,
        "judge_rerun_tag": JUDGE_RERUN_TAG,
        "temperature_sent": False,
        "batch_id": batch.id,
        "input_file_id": batch_input_file.id,
        "status_at_submission": batch.status,
        "submitted_at_utc": now_iso(),
        "batch_jsonl_path": str(batch_jsonl_path),
        "plan_path": str(plan_path),
        "data_root": str(DATA_ROOT),
    }

    manifest_path = manifest_dir / f"slogan_judge_batch_manifest__{JUDGE_RERUN_TAG}__{batch.id}.json"
    if manifest_path.exists():
        raise FileExistsError(f"Refusing to overwrite manifest: {manifest_path}")

    write_json(manifest_path, json_safe(batch_info))
    batch_info["manifest_path"] = str(manifest_path)

    print("Submitted batch:")
    print(json.dumps(json_safe(batch_info), indent=2))
    return batch_info


judge_batch_info = submit_openai_batch(
    batch_jsonl_path=batch_jsonl_path,
    plan_path=plan_csv_path,
    manifest_dir=DIRS["manifests"],
)

judge_batch_info

Submitted batch:
{
  "run_id": "20260525_030656__bf5164cf",
  "experiment_id": "slogan_llm_judge_gpt54_creativity",
  "stage": "slogan_llm_judge",
  "judge_provider": "openai",
  "judge_model": "gpt-5.4",
  "judge_rerun_tag": "notemp_reasonlow_maxtok256_fullrerun",
  "temperature_sent": false,
  "batch_id": "batch_6a13f51b58d08190a828ae6009e40c42",
  "input_file_id": "file-NAbEeTH8hZzCBFkEU2eA6s",
  "status_at_submission": "validating",
  "submitted_at_utc": "2026-05-25T07:07:07.525092+00:00",
  "batch_jsonl_path": "ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/03_batch_inputs/slogan_judge_batch_input__notemp_reasonlow_maxtok256_fullrerun__20260525_030656__bf5164cf.jsonl",
  "plan_path": "ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/02_plans/slogan_judge_plan__notemp_reasonlow_maxtok256_fullrerun__20260525_030656__bf5164cf.csv",
  "data_root": "ai_data/

{'run_id': '20260525_030656__bf5164cf',
 'experiment_id': 'slogan_llm_judge_gpt54_creativity',
 'stage': 'slogan_llm_judge',
 'judge_provider': 'openai',
 'judge_model': 'gpt-5.4',
 'judge_rerun_tag': 'notemp_reasonlow_maxtok256_fullrerun',
 'temperature_sent': False,
 'batch_id': 'batch_6a13f51b58d08190a828ae6009e40c42',
 'input_file_id': 'file-NAbEeTH8hZzCBFkEU2eA6s',
 'status_at_submission': 'validating',
 'submitted_at_utc': '2026-05-25T07:07:07.525092+00:00',
 'batch_jsonl_path': 'ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/03_batch_inputs/slogan_judge_batch_input__notemp_reasonlow_maxtok256_fullrerun__20260525_030656__bf5164cf.jsonl',
 'plan_path': 'ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/02_plans/slogan_judge_plan__notemp_reasonlow_maxtok256_fullrerun__20260525_030656__bf5164cf.csv',
 'data_root': 'ai_data/deflect_creativity/judging/judge_

In [101]:
def check_openai_batch(batch_id: str) -> dict:
    batch = client.batches.retrieve(batch_id)

    info = {
        "batch_id": batch.id,
        "status": batch.status,
        "request_counts": None,
        "output_file_id": batch.output_file_id,
        "error_file_id": batch.error_file_id,
        "created_at": batch.created_at,
        "in_progress_at": getattr(batch, "in_progress_at", None),
        "finalizing_at": getattr(batch, "finalizing_at", None),
        "completed_at": getattr(batch, "completed_at", None),
        "failed_at": getattr(batch, "failed_at", None),
        "expired_at": getattr(batch, "expired_at", None),
        "cancelled_at": getattr(batch, "cancelled_at", None),
    }

    if batch.request_counts:
        info["request_counts"] = {
            "total": batch.request_counts.total,
            "completed": batch.request_counts.completed,
            "failed": batch.request_counts.failed,
        }

    usage = getattr(batch, "usage", None)
    if usage:
        try:
            info["usage"] = usage.model_dump()
        except Exception:
            info["usage"] = str(usage)

    print(json.dumps(json_safe(info), indent=2))
    return info

In [157]:
judge_status = check_openai_batch(judge_batch_info["batch_id"])
judge_status

{
  "batch_id": "batch_6a13f51b58d08190a828ae6009e40c42",
  "status": "completed",
  "request_counts": {
    "total": 16200,
    "completed": 16200,
    "failed": 0
  },
  "output_file_id": "file-Fa9M94WTQN6pBuXTQJBdxN",
  "error_file_id": "file-UoLCamf7pwqToLNWTSn7hi",
  "created_at": 1779692827,
  "in_progress_at": 1779692831,
  "finalizing_at": 1779694862,
  "completed_at": 1779699524,
  "failed_at": null,
  "expired_at": null,
  "cancelled_at": null,
  "usage": {
    "input_tokens": 8041859,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens": 941578,
    "output_tokens_details": {
      "reasoning_tokens": 828185
    },
    "total_tokens": 8983437
  }
}


{'batch_id': 'batch_6a13f51b58d08190a828ae6009e40c42',
 'status': 'completed',
 'request_counts': {'total': 16200, 'completed': 16200, 'failed': 0},
 'output_file_id': 'file-Fa9M94WTQN6pBuXTQJBdxN',
 'error_file_id': 'file-UoLCamf7pwqToLNWTSn7hi',
 'created_at': 1779692827,
 'in_progress_at': 1779692831,
 'finalizing_at': 1779694862,
 'completed_at': 1779699524,
 'failed_at': None,
 'expired_at': None,
 'cancelled_at': None,
 'usage': {'input_tokens': 8041859,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 941578,
  'output_tokens_details': {'reasoning_tokens': 828185},
  'total_tokens': 8983437}}

In [89]:
# def download_batch_error_file_if_available(batch_id: str, out_dir: Path) -> Optional[Path]:
#     batch = client.batches.retrieve(batch_id)

#     print({
#         "batch_id": batch.id,
#         "status": batch.status,
#         "request_counts": batch.request_counts.model_dump() if batch.request_counts else None,
#         "output_file_id": batch.output_file_id,
#         "error_file_id": batch.error_file_id,
#     })

#     if not batch.error_file_id:
#         print("No error_file_id available yet.")
#         return None

#     out_dir.mkdir(parents=True, exist_ok=True)
#     error_path = out_dir / f"slogan_judge__{batch_id}__errors.jsonl"

#     file_response = client.files.content(batch.error_file_id)
#     error_path.write_text(file_response.text, encoding="utf-8")

#     print("Downloaded:", error_path)

#     errors = read_jsonl(error_path)
#     print("Number of error records:", len(errors))

#     for rec in errors[:10]:
#         print(json.dumps(rec, indent=2)[:3000])
#         print("-" * 80)

#     return error_path


# bad_error_path = download_batch_error_file_if_available(
#     judge_batch_info["batch_id"],
#     DIRS["raw_errors"],
# )

{'batch_id': 'batch_6a13e5ee7f6881908771b0f6545cef6f', 'status': 'completed', 'request_counts': {'completed': 2, 'failed': 16198, 'total': 16200}, 'output_file_id': None, 'error_file_id': 'file-EsR69tWZceW6Mf6m6kiu6v'}
Downloaded: ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_020129__a13bca1e/06_raw_errors/slogan_judge__batch_6a13e5ee7f6881908771b0f6545cef6f__errors.jsonl
Number of error records: 16200
{
  "id": "batch_req_6a13eedae34881908debcb2024edf294",
  "custom_id": "sloganjudge__b504a80305409898a6f909bc391d29e0",
  "response": {
    "status_code": 400,
    "request_id": "12b403b6-617b-4c2f-bc05-3d98b783a298",
    "body": {
      "error": {
        "message": "Unsupported parameter: 'temperature' is not supported with this model.",
        "type": "invalid_request_error",
        "param": "temperature",
        "code": null
      }
    }
  },
  "error": null
}
-----------------------------------------------------------------------

In [ ]:
# Optional reload after kernel restart:
# manifest_path = Path("ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_.../04_manifests/slogan_judge_batch_manifest__batch_....json")
# judge_batch_info = read_json(manifest_path)
# DATA_ROOT = Path(judge_batch_info["data_root"])
# DIRS = {
#     "metadata": DATA_ROOT / "00_metadata",
#     "inputs": DATA_ROOT / "01_inputs",
#     "plans": DATA_ROOT / "02_plans",
#     "batch_inputs": DATA_ROOT / "03_batch_inputs",
#     "manifests": DATA_ROOT / "04_manifests",
#     "raw_outputs": DATA_ROOT / "05_raw_outputs",
#     "raw_errors": DATA_ROOT / "06_raw_errors",
#     "parsed": DATA_ROOT / "07_parsed",
#     "compiled": DATA_ROOT / "08_compiled",
# }
# judge_batch_info

In [158]:
def download_openai_batch_results(
    batch_id: str,
    raw_output_dir: Path,
    raw_error_dir: Path,
) -> tuple[Optional[Path], Optional[Path]]:
    batch = client.batches.retrieve(batch_id)

    if batch.status != "completed":
        raise RuntimeError(f"Batch is not completed yet. Current status: {batch.status}")

    output_path = raw_output_dir / f"slogan_judge__{batch_id}__output.jsonl"
    error_path = raw_error_dir / f"slogan_judge__{batch_id}__errors.jsonl"

    if output_path.exists():
        raise FileExistsError(f"Refusing to overwrite output file: {output_path}")

    if batch.output_file_id:
        file_response = client.files.content(batch.output_file_id)
        output_path.write_text(file_response.text, encoding="utf-8")
        print(f"Downloaded output: {output_path}")
    else:
        output_path = None
        print("No output file.")

    if batch.error_file_id:
        if error_path.exists():
            raise FileExistsError(f"Refusing to overwrite error file: {error_path}")
        error_response = client.files.content(batch.error_file_id)
        error_path.write_text(error_response.text, encoding="utf-8")
        print(f"Downloaded errors: {error_path}")
    else:
        error_path = None
        print("No error file.")

    return output_path, error_path


judge_output_path, judge_error_path = download_openai_batch_results(
    batch_id=judge_batch_info["batch_id"],
    raw_output_dir=DIRS["raw_outputs"],
    raw_error_dir=DIRS["raw_errors"],
)

judge_output_path, judge_error_path

Downloaded output: ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/05_raw_outputs/slogan_judge__batch_6a13f51b58d08190a828ae6009e40c42__output.jsonl
Downloaded errors: ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/06_raw_errors/slogan_judge__batch_6a13f51b58d08190a828ae6009e40c42__errors.jsonl


(PosixPath('ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/05_raw_outputs/slogan_judge__batch_6a13f51b58d08190a828ae6009e40c42__output.jsonl'),
 PosixPath('ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/06_raw_errors/slogan_judge__batch_6a13f51b58d08190a828ae6009e40c42__errors.jsonl'))

In [159]:
def extract_text_from_responses_api_body(body: dict) -> str:
    if not isinstance(body, dict):
        return ""

    if body.get("output_text"):
        return str(body["output_text"]).strip()

    texts = []
    for item in body.get("output", []) or []:
        for content in item.get("content", []) or []:
            if isinstance(content, dict) and content.get("type") in {"output_text", "text"} and "text" in content:
                texts.append(content["text"])

    return "\n".join(texts).strip()


def flatten_usage(usage: Optional[dict]) -> dict:
    usage = usage or {}
    input_details = usage.get("input_tokens_details") or {}
    output_details = usage.get("output_tokens_details") or {}

    return {
        "usage_input_tokens": usage.get("input_tokens"),
        "usage_output_tokens": usage.get("output_tokens"),
        "usage_total_tokens": usage.get("total_tokens"),
        "usage_cached_tokens": input_details.get("cached_tokens"),
        "usage_reasoning_tokens": output_details.get("reasoning_tokens"),
    }


def parse_integer_score(raw_text: str) -> Optional[int]:
    if raw_text is None:
        return None

    text = str(raw_text).strip()

    if re.fullmatch(r"[1-5]", text):
        return int(text)

    digits = re.findall(r"\b[1-5]\b", text)
    all_digits = re.findall(r"\d", text)

    if len(digits) == 1 and len(all_digits) == 1:
        return int(digits[0])

    return None


def parse_judge_batch_output(
    batch_output_path: Path,
    plan_path: Path,
    parsed_dir: Path,
    batch_id: str,
) -> dict:
    if batch_output_path is None or not Path(batch_output_path).exists():
        raise FileNotFoundError(f"Missing batch output path: {batch_output_path}")

    plan_df = pd.read_csv(plan_path)
    plan_by_key = {row["judge_request_key"]: row.to_dict() for _, row in plan_df.iterrows()}

    batch_records = read_jsonl(batch_output_path)

    parsed_jsonl_path = parsed_dir / f"slogan_judge__{batch_id}__parsed.jsonl"
    parsed_csv_path = parsed_dir / f"slogan_judge__{batch_id}__parsed.csv"
    parsed_pkl_path = parsed_dir / f"slogan_judge__{batch_id}__parsed.pkl"

    if parsed_jsonl_path.exists() or parsed_csv_path.exists() or parsed_pkl_path.exists():
        raise FileExistsError("Refusing to overwrite existing parsed files.")

    parsed_records = []

    for rec in batch_records:
        judge_request_key = rec.get("custom_id")
        plan_row = plan_by_key.get(judge_request_key, {})
        response = rec.get("response") or {}
        error = rec.get("error")

        if response and response.get("body"):
            body = response["body"]
            raw_text = clean_model_text(extract_text_from_responses_api_body(body))
            usage = body.get("usage") or {}
            score = parse_integer_score(raw_text)

            status = "success" if score in {1, 2, 3, 4, 5} else "parse_error"

            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "judge_status": status,
                "judge_raw_text": raw_text,
                "slogan_creativity_1to5": score,
                "provider_response_id": body.get("id"),
                "usage": usage,
                **flatten_usage(usage),
                "error": None if status == "success" else f"Could not parse integer score from: {raw_text!r}",
                "batch_custom_id": judge_request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
            }
        else:
            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "judge_status": "api_error",
                "judge_raw_text": None,
                "slogan_creativity_1to5": None,
                "provider_response_id": None,
                "usage": None,
                **flatten_usage(None),
                "error": error,
                "batch_custom_id": judge_request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
            }

        parsed_records.append(record)

        with open(parsed_jsonl_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(json_safe(record), ensure_ascii=False) + "\n")

    parsed_df = pd.DataFrame(parsed_records)
    parsed_df.to_csv(parsed_csv_path, index=False)
    parsed_df.to_pickle(parsed_pkl_path)

    summary = {
        "batch_id": batch_id,
        "n_records": len(parsed_df),
        "n_success": int((parsed_df["judge_status"] == "success").sum()),
        "n_parse_error": int((parsed_df["judge_status"] == "parse_error").sum()),
        "n_api_error": int((parsed_df["judge_status"] == "api_error").sum()),
        "parsed_jsonl_path": str(parsed_jsonl_path),
        "parsed_csv_path": str(parsed_csv_path),
        "parsed_pkl_path": str(parsed_pkl_path),
    }

    summary_path = parsed_dir / f"slogan_judge__{batch_id}__parse_summary.json"
    write_json(summary_path, json_safe(summary))

    print(json.dumps(json_safe(summary), indent=2))
    return summary


judge_parse_summary = parse_judge_batch_output(
    batch_output_path=judge_output_path,
    plan_path=Path(judge_batch_info["plan_path"]),
    parsed_dir=DIRS["parsed"],
    batch_id=judge_batch_info["batch_id"],
)

judge_scores_df = pd.read_pickle(judge_parse_summary["parsed_pkl_path"])

display(judge_scores_df["judge_status"].value_counts(dropna=False))
display(judge_scores_df["slogan_creativity_1to5"].value_counts(dropna=False).sort_index())

{
  "batch_id": "batch_6a13f51b58d08190a828ae6009e40c42",
  "n_records": 16199,
  "n_success": 16199,
  "n_parse_error": 0,
  "n_api_error": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/07_parsed/slogan_judge__batch_6a13f51b58d08190a828ae6009e40c42__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/07_parsed/slogan_judge__batch_6a13f51b58d08190a828ae6009e40c42__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/07_parsed/slogan_judge__batch_6a13f51b58d08190a828ae6009e40c42__parsed.pkl"
}


judge_status
success    16199
Name: count, dtype: int64

slogan_creativity_1to5
1       45
2      561
3     1912
4    12526
5     1155
Name: count, dtype: int64

In [160]:
failed = judge_scores_df[judge_scores_df["judge_status"].ne("success")].copy()

print("Failures:", len(failed))

if len(failed):
    display(
        failed[
            [
                "judge_request_key",
                "provider",
                "model",
                "task_id",
                "method",
                "strategy",
                "slogan_text",
                "judge_raw_text",
                "error",
            ]
        ].head(100)
    )

Failures: 0


In [171]:
merge_cols = [
    "row_uid",
    "judge_request_key",
    "judge_status",
    "judge_raw_text",
    "slogan_creativity_1to5",
    "provider_response_id",
    "usage_input_tokens",
    "usage_output_tokens",
    "usage_total_tokens",
    "usage_cached_tokens",
    "usage_reasoning_tokens",
    "batch_id",
]
merge_cols = [c for c in merge_cols if c in judge_scores_df.columns]

scores_small = judge_scores_df[merge_cols].copy()

if scores_small["row_uid"].duplicated().any():
    dupes = scores_small[scores_small["row_uid"].duplicated(keep=False)].sort_values("row_uid")
    display(dupes.head(20))
    raise RuntimeError("Duplicate row_uid in judge scores.")

scored_slogans_df = slogan_df.merge(
    scores_small,
    on="row_uid",
    how="left",
    validate="one_to_one",
)

if len(scored_slogans_df) != len(slogan_df):
    raise RuntimeError("Merge changed row count.")

missing_scores = scored_slogans_df["slogan_creativity_1to5"].isna().sum()

print("Rows:", len(scored_slogans_df))
print("Missing scores:", missing_scores)

scored_csv_path = DIRS["compiled"] / f"slogans_with_gpt54_creativity_scores__{RUN_ID}.csv"
scored_pkl_path = DIRS["compiled"] / f"slogans_with_gpt54_creativity_scores__{RUN_ID}.pkl"

scored_slogans_df.to_csv(scored_csv_path, index=False)
scored_slogans_df.to_pickle(scored_pkl_path)

print(scored_csv_path)
print(scored_pkl_path)

display(scored_slogans_df.head())

Rows: 16200
Missing scores: 1
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogans_with_gpt54_creativity_scores__20260525_030656__bf5164cf.csv
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogans_with_gpt54_creativity_scores__20260525_030656__bf5164cf.pkl


,_old_row_pos,agent_id,agent_index,anchor_count,batch_custom_id,batch_id_x,batch_name,batch_output_file,condition,context_count,...,judge_status,judge_raw_text,slogan_creativity_1to5,provider_response_id_y,usage_input_tokens_y,usage_output_tokens_y,usage_total_tokens_y,usage_cached_tokens_y,usage_reasoning_tokens_y,batch_id_y
0,0.0,base_001__a1,1.0,0,r1__ae37e6a4459629b922a98b61,batch_6a0a608e99408190b15d321761c983a7,NaN,ai_data/deflect_creativity/openai/model_gpt-5....,base,NaN,...,success,4,4.0,resp_07fda711882dfebe006a13f54267f88197ae1e37c...,496.0,71.0,567.0,0.0,64.0,batch_6a13f51b58d08190a828ae6009e40c42
1,1.0,base_001__a1,1.0,0,r2__4523e53313d039e7622c16a1,batch_6a0a66ff75148190916b217bd42d0379,NaN,ai_data/deflect_creativity/openai/model_gpt-5....,base,NaN,...,success,4,4.0,resp_059eca0db148e8b3006a13f545981481969d479ee...,495.0,48.0,543.0,0.0,41.0,batch_6a13f51b58d08190a828ae6009e40c42
2,2.0,base_002__a1,1.0,0,r1__8cb17f81c524a85271342c9d,batch_6a0a608e99408190b15d321761c983a7,NaN,ai_data/deflect_creativity/openai/model_gpt-5....,base,NaN,...,success,4,4.0,resp_0bdd0504fd75f6d2006a13f542dab88194a998e30...,496.0,60.0,556.0,0.0,53.0,batch_6a13f51b58d08190a828ae6009e40c42
3,3.0,base_002__a1,1.0,0,r2__d6ecab12e1f888e7b2aa16df,batch_6a0a66ff75148190916b217bd42d0379,NaN,ai_data/deflect_creativity/openai/model_gpt-5....,base,NaN,...,success,4,4.0,resp_0b2d971bbbd7b7dd006a13f546e8e48196bcac4c2...,493.0,51.0,544.0,0.0,44.0,batch_6a13f51b58d08190a828ae6009e40c42
4,4.0,base_003__a1,1.0,0,r1__57bc8ea5d603182f310443f0,batch_6a0a608e99408190b15d321761c983a7,NaN,ai_data/deflect_creativity/openai/model_gpt-5....,base,NaN,...,success,4,4.0,resp_0ce11293a5bd5e74006a13f548857c81948d5d618...,497.0,54.0,551.0,0.0,47.0,batch_6a13f51b58d08190a828ae6009e40c42


In [172]:
# Cell 17 — Cell-level summaries, without token usage columns

score_summary = (
    scored_slogans_df
    .groupby(["provider", "model", "task_id", "method", "strategy"], observed=True)
    .agg(
        n=("slogan_creativity_1to5", "size"),
        n_scored=("slogan_creativity_1to5", lambda x: x.notna().sum()),
        mean_slogan_creativity=("slogan_creativity_1to5", "mean"),
        sd_slogan_creativity=("slogan_creativity_1to5", "std"),
        min_slogan_creativity=("slogan_creativity_1to5", "min"),
        max_slogan_creativity=("slogan_creativity_1to5", "max"),
    )
    .reset_index()
)

summary_csv_path = DIRS["compiled"] / f"slogan_creativity_summary_by_cell__{RUN_ID}.csv"
summary_pkl_path = DIRS["compiled"] / f"slogan_creativity_summary_by_cell__{RUN_ID}.pkl"

score_summary.to_csv(summary_csv_path, index=False)
score_summary.to_pickle(summary_pkl_path)

display(score_summary.sort_values(["provider", "task_id", "method", "strategy"]))

print(summary_csv_path)
print(summary_pkl_path)

# Optional sanity checks
print("Rows:", len(scored_slogans_df))
print("Scores missing:", scored_slogans_df["slogan_creativity_1to5"].isna().sum())
print("Score distribution:")
display(scored_slogans_df["slogan_creativity_1to5"].value_counts(dropna=False).sort_index())

,provider,model,task_id,method,strategy,n,n_scored,mean_slogan_creativity,sd_slogan_creativity,min_slogan_creativity,max_slogan_creativity
0,anthropic,claude-sonnet-4-6,slogan_blood_donation,indep,diverge,150,150,4.340000,0.541313,2.0,5.0
1,anthropic,claude-sonnet-4-6,slogan_blood_donation,indep,vanilla,150,150,4.386667,0.540776,2.0,5.0
2,anthropic,claude-sonnet-4-6,slogan_blood_donation,peer1,diverge,150,150,4.140000,0.555992,2.0,5.0
3,anthropic,claude-sonnet-4-6,slogan_blood_donation,peer1,vanilla,150,150,4.126667,0.508936,2.0,5.0
4,anthropic,claude-sonnet-4-6,slogan_blood_donation,peer2,diverge,150,150,4.340000,0.553573,2.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...
103,openai,gpt-5.4,slogan_soda,repr,vanilla,150,150,3.953333,0.241268,2.0,4.0
104,openai,gpt-5.4,slogan_soda,self,diverge,150,150,3.720000,0.686593,1.0,5.0
105,openai,gpt-5.4,slogan_soda,self,vanilla,150,150,3.853333,0.583747,1.0,5.0
106,openai,gpt-5.4,slogan_soda,strat,diverge,150,150,3.613333,0.642840,1.0,4.0


ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogan_creativity_summary_by_cell__20260525_030656__bf5164cf.csv
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogan_creativity_summary_by_cell__20260525_030656__bf5164cf.pkl
Rows: 16200
Scores missing: 1
Score distribution:


slogan_creativity_1to5
1.0       45
2.0      561
3.0     1912
4.0    12526
5.0     1155
NaN        1
Name: count, dtype: int64

In [173]:
# Inspect the one missing score
missing_score = scored_slogans_df[scored_slogans_df["slogan_creativity_1to5"].isna()].copy()

print("Missing rows:", len(missing_score))

cols_to_show = [
    "provider", "model", "task_id", "method", "strategy",
    "slogan_text", "judge_status", "judge_raw_text", "error",
    "judge_request_key", "batch_id"
]
cols_to_show = [c for c in cols_to_show if c in missing_score.columns]

display(missing_score[cols_to_show])

Missing rows: 1


,provider,model,task_id,method,strategy,slogan_text,judge_status,judge_raw_text,error,judge_request_key
10730,gemini,gemini-2.5-pro,slogan_blood_donation,peer2,diverge,Your blood is someone's first breath.,NaN,NaN,NaN,NaN


In [174]:
missing_row = scored_slogans_df[scored_slogans_df["slogan_creativity_1to5"].isna()].copy()

row_uid = missing_row["row_uid"].iloc[0]
print("Missing row_uid:", row_uid)

print("\nIn slogan_df?")
print((slogan_df["row_uid"] == row_uid).sum())

print("\nIn judge_plan_df?")
print((judge_plan_df["row_uid"] == row_uid).sum() if "row_uid" in judge_plan_df.columns else "judge_plan_df has no row_uid")

print("\nIn judge_scores_df?")
print((judge_scores_df["row_uid"] == row_uid).sum() if "row_uid" in judge_scores_df.columns else "judge_scores_df has no row_uid")

display(missing_row[
    ["row_uid", "provider", "model", "task_id", "method", "strategy", "slogan_text"]
])

Missing row_uid: 5f12aea18ab4a21312423419883c2379

In slogan_df?
1

In judge_plan_df?
1

In judge_scores_df?
0


,row_uid,provider,model,task_id,method,strategy,slogan_text
10730,5f12aea18ab4a21312423419883c2379,gemini,gemini-2.5-pro,slogan_blood_donation,peer2,diverge,Your blood is someone's first breath.


In [175]:
plan_match = judge_plan_df[judge_plan_df["row_uid"] == row_uid].copy()

print("Plan matches:", len(plan_match))
display(plan_match[
    ["row_uid", "judge_request_key", "provider", "model", "task_id", "method", "strategy", "slogan_text"]
])

Plan matches: 1


,row_uid,judge_request_key,provider,model,task_id,method,strategy,slogan_text
10730,5f12aea18ab4a21312423419883c2379,sloganjudge__eb26065d9fd1825cf43079863dc31dfd,gemini,gemini-2.5-pro,slogan_blood_donation,peer2,diverge,Your blood is someone's first breath.


In [176]:
# Check whether the missing request exists in the raw output file

req_key = "sloganjudge__eb26065d9fd1825cf43079863dc31dfd"

raw_records = read_jsonl(judge_output_path)

raw_match = [r for r in raw_records if r.get("custom_id") == req_key]

print("Raw output records:", len(raw_records))
print("Raw matches for missing request:", len(raw_match))

if raw_match:
    print(json.dumps(raw_match[0], indent=2)[:5000])

Raw output records: 16199
Raw matches for missing request: 0


In [177]:
# Check whether the missing request exists in the error file for the successful batch

batch = client.batches.retrieve(judge_batch_info["batch_id"])

print({
    "batch_id": batch.id,
    "status": batch.status,
    "output_file_id": batch.output_file_id,
    "error_file_id": batch.error_file_id,
    "request_counts": batch.request_counts.model_dump() if batch.request_counts else None,
})

if batch.error_file_id:
    err_path = DIRS["raw_errors"] / f"slogan_judge__{batch.id}__errors.jsonl"
    if not err_path.exists():
        err_response = client.files.content(batch.error_file_id)
        err_path.write_text(err_response.text, encoding="utf-8")

    err_records = read_jsonl(err_path)
    err_match = [r for r in err_records if r.get("custom_id") == req_key]

    print("Error records:", len(err_records))
    print("Error matches for missing request:", len(err_match))

    if err_match:
        print(json.dumps(err_match[0], indent=2)[:5000])
else:
    print("No error_file_id for this batch.")

{'batch_id': 'batch_6a13f51b58d08190a828ae6009e40c42', 'status': 'completed', 'output_file_id': 'file-Fa9M94WTQN6pBuXTQJBdxN', 'error_file_id': 'file-UoLCamf7pwqToLNWTSn7hi', 'request_counts': {'completed': 16200, 'failed': 0, 'total': 16200}}
Error records: 1
Error matches for missing request: 1
{
  "id": "batch_req_6a140dfa641881908867c05edb84a917",
  "custom_id": "sloganjudge__eb26065d9fd1825cf43079863dc31dfd",
  "response": {
    "status_code": 500,
    "request_id": "",
    "body": {
      "error": {
        "message": "BatchAPI failed to execute task in batch"
      }
    }
  },
  "error": null
}


In [178]:
# Final fallback: flagged within-cell median imputation for exactly this missing planned request

score_col = "slogan_creativity_1to5"
cell_cols = ["provider", "model", "task_id", "method", "strategy"]

if "slogan_creativity_imputed" not in scored_slogans_df.columns:
    scored_slogans_df["slogan_creativity_imputed"] = False

missing_mask = scored_slogans_df[score_col].isna()

assert missing_mask.sum() == 1, f"Expected exactly 1 missing score, found {missing_mask.sum()}"

cell_medians = (
    scored_slogans_df
    .groupby(cell_cols, observed=True)[score_col]
    .transform("median")
)

scored_slogans_df.loc[missing_mask, score_col] = cell_medians[missing_mask]
scored_slogans_df.loc[missing_mask, "slogan_creativity_imputed"] = True

display(
    scored_slogans_df.loc[
        scored_slogans_df["slogan_creativity_imputed"],
        cell_cols + ["slogan_text", score_col, "slogan_creativity_imputed"]
    ]
)

print("Remaining missing:", scored_slogans_df[score_col].isna().sum())

,provider,model,task_id,method,strategy,slogan_text,slogan_creativity_1to5,slogan_creativity_imputed
10730,gemini,gemini-2.5-pro,slogan_blood_donation,peer2,diverge,Your blood is someone's first breath.,4.0,True


Remaining missing: 0


In [179]:
analysis_df = scored_slogans_df.copy()

analysis_df["slogan_creativity_1to5"] = pd.to_numeric(
    analysis_df["slogan_creativity_1to5"],
    errors="coerce",
)

task_stats = (
    analysis_df
    .groupby("task_id", observed=True)["slogan_creativity_1to5"]
    .agg(["mean", "std"])
    .rename(columns={"mean": "task_score_mean", "std": "task_score_sd"})
    .reset_index()
)

analysis_df = analysis_df.merge(task_stats, on="task_id", how="left", validate="many_to_one")

analysis_df["slogan_creativity_z"] = (
    analysis_df["slogan_creativity_1to5"] - analysis_df["task_score_mean"]
) / analysis_df["task_score_sd"]

analysis_df.loc[analysis_df["task_score_sd"].eq(0), "slogan_creativity_z"] = np.nan

analysis_csv_path = DIRS["compiled"] / f"slogans_with_gpt54_creativity_scores_z__{RUN_ID}.csv"
analysis_pkl_path = DIRS["compiled"] / f"slogans_with_gpt54_creativity_scores_z__{RUN_ID}.pkl"

analysis_df.to_csv(analysis_csv_path, index=False)
analysis_df.to_pickle(analysis_pkl_path)

print(analysis_csv_path)
print(analysis_pkl_path)

display(
    analysis_df
    .groupby("task_id", observed=True)
    .agg(
        n=("slogan_creativity_1to5", "size"),
        mean_raw=("slogan_creativity_1to5", "mean"),
        sd_raw=("slogan_creativity_1to5", "std"),
        mean_z=("slogan_creativity_z", "mean"),
        sd_z=("slogan_creativity_z", "std"),
    )
    .reset_index()
)

ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogans_with_gpt54_creativity_scores_z__20260525_030656__bf5164cf.csv
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogans_with_gpt54_creativity_scores_z__20260525_030656__bf5164cf.pkl


,task_id,n,mean_raw,sd_raw,mean_z,sd_z
0,slogan_blood_donation,5400,4.000926,0.661288,2.368887e-16,1.0
1,slogan_smartphone,5400,3.844630,0.505323,2.390063e-17,1.0
2,slogan_soda,5400,3.781481,0.542361,-3.352462e-16,1.0


In [181]:
# Cell A — Save imputed row-level judged slogans

# Ensure imputation flag exists
if "slogan_creativity_imputed" not in scored_slogans_df.columns:
    scored_slogans_df["slogan_creativity_imputed"] = False

scored_slogans_df["slogan_creativity_imputed"] = (
    scored_slogans_df["slogan_creativity_imputed"]
    .fillna(False)
    .astype(bool)
)

# Ensure imputation_note exists
if "imputation_note" not in scored_slogans_df.columns:
    scored_slogans_df["imputation_note"] = ""

scored_slogans_df.loc[
    scored_slogans_df["slogan_creativity_imputed"] & scored_slogans_df["imputation_note"].astype(str).eq(""),
    "imputation_note"
] = "BatchAPI 500 execution failure; imputed with provider-model-task-method-strategy cell median."

scored_csv_path = DIRS["compiled"] / f"slogans_with_gpt54_creativity_scores_IMPUTED__{RUN_ID}.csv"
scored_pkl_path = DIRS["compiled"] / f"slogans_with_gpt54_creativity_scores_IMPUTED__{RUN_ID}.pkl"

scored_slogans_df.to_csv(scored_csv_path, index=False)
scored_slogans_df.to_pickle(scored_pkl_path)

print("Saved imputed row-level scored slogans:")
print(scored_csv_path)
print(scored_pkl_path)

print("\nRows:", len(scored_slogans_df))
print("Missing scores:", scored_slogans_df["slogan_creativity_1to5"].isna().sum())
print("Imputed rows:", int(scored_slogans_df["slogan_creativity_imputed"].sum()))

display_cols = [
    "provider", "model", "task_id", "method", "strategy",
    "slogan_text", "slogan_creativity_1to5",
    "slogan_creativity_imputed", "imputation_note"
]
display_cols = [c for c in display_cols if c in scored_slogans_df.columns]

display(
    scored_slogans_df.loc[
        scored_slogans_df["slogan_creativity_imputed"],
        display_cols
    ]
)

Saved imputed row-level scored slogans:
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogans_with_gpt54_creativity_scores_IMPUTED__20260525_030656__bf5164cf.csv
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogans_with_gpt54_creativity_scores_IMPUTED__20260525_030656__bf5164cf.pkl

Rows: 16200
Missing scores: 0
Imputed rows: 1


,provider,model,task_id,method,strategy,slogan_text,slogan_creativity_1to5,slogan_creativity_imputed,imputation_note
10730,gemini,gemini-2.5-pro,slogan_blood_donation,peer2,diverge,Your blood is someone's first breath.,4.0,True,BatchAPI 500 execution failure; imputed with p...


In [182]:
# Cell B — Cell-level summaries after imputation

if "slogan_creativity_imputed" not in scored_slogans_df.columns:
    scored_slogans_df["slogan_creativity_imputed"] = False

scored_slogans_df["slogan_creativity_imputed"] = (
    scored_slogans_df["slogan_creativity_imputed"]
    .fillna(False)
    .astype(bool)
)

score_summary = (
    scored_slogans_df
    .groupby(["provider", "model", "task_id", "method", "strategy"], observed=True)
    .agg(
        n=("slogan_creativity_1to5", "size"),
        n_scored=("slogan_creativity_1to5", lambda x: x.notna().sum()),
        n_imputed=("slogan_creativity_imputed", lambda x: int(x.sum())),
        mean_slogan_creativity=("slogan_creativity_1to5", "mean"),
        sd_slogan_creativity=("slogan_creativity_1to5", "std"),
        min_slogan_creativity=("slogan_creativity_1to5", "min"),
        max_slogan_creativity=("slogan_creativity_1to5", "max"),
    )
    .reset_index()
)

summary_csv_path = DIRS["compiled"] / f"slogan_creativity_summary_by_cell_IMPUTED__{RUN_ID}.csv"
summary_pkl_path = DIRS["compiled"] / f"slogan_creativity_summary_by_cell_IMPUTED__{RUN_ID}.pkl"

score_summary.to_csv(summary_csv_path, index=False)
score_summary.to_pickle(summary_pkl_path)

display(score_summary.sort_values(["provider", "task_id", "method", "strategy"]))

print("Saved cell summaries:")
print(summary_csv_path)
print(summary_pkl_path)

print("\nSummary rows:", len(score_summary))
print("Expected summary rows:", 3 * 3 * 6 * 2)
print("Any n != 150?")
display(score_summary[score_summary["n"] != 150])

,provider,model,task_id,method,strategy,n,n_scored,n_imputed,mean_slogan_creativity,sd_slogan_creativity,min_slogan_creativity,max_slogan_creativity
0,anthropic,claude-sonnet-4-6,slogan_blood_donation,indep,diverge,150,150,0,4.340000,0.541313,2.0,5.0
1,anthropic,claude-sonnet-4-6,slogan_blood_donation,indep,vanilla,150,150,0,4.386667,0.540776,2.0,5.0
2,anthropic,claude-sonnet-4-6,slogan_blood_donation,peer1,diverge,150,150,0,4.140000,0.555992,2.0,5.0
3,anthropic,claude-sonnet-4-6,slogan_blood_donation,peer1,vanilla,150,150,0,4.126667,0.508936,2.0,5.0
4,anthropic,claude-sonnet-4-6,slogan_blood_donation,peer2,diverge,150,150,0,4.340000,0.553573,2.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...
103,openai,gpt-5.4,slogan_soda,repr,vanilla,150,150,0,3.953333,0.241268,2.0,4.0
104,openai,gpt-5.4,slogan_soda,self,diverge,150,150,0,3.720000,0.686593,1.0,5.0
105,openai,gpt-5.4,slogan_soda,self,vanilla,150,150,0,3.853333,0.583747,1.0,5.0
106,openai,gpt-5.4,slogan_soda,strat,diverge,150,150,0,3.613333,0.642840,1.0,4.0


Saved cell summaries:
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogan_creativity_summary_by_cell_IMPUTED__20260525_030656__bf5164cf.csv
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogan_creativity_summary_by_cell_IMPUTED__20260525_030656__bf5164cf.pkl

Summary rows: 108
Expected summary rows: 108
Any n != 150?


,provider,model,task_id,method,strategy,n,n_scored,n_imputed,mean_slogan_creativity,sd_slogan_creativity,min_slogan_creativity,max_slogan_creativity


In [183]:
# Cell C — Add within-task z-score for slogan creativity

analysis_df = scored_slogans_df.copy()

analysis_df["slogan_creativity_1to5"] = pd.to_numeric(
    analysis_df["slogan_creativity_1to5"],
    errors="coerce",
)

task_stats = (
    analysis_df
    .groupby("task_id", observed=True)["slogan_creativity_1to5"]
    .agg(["mean", "std"])
    .rename(columns={"mean": "task_score_mean", "std": "task_score_sd"})
    .reset_index()
)

analysis_df = analysis_df.merge(task_stats, on="task_id", how="left", validate="many_to_one")

analysis_df["slogan_creativity_z"] = (
    analysis_df["slogan_creativity_1to5"] - analysis_df["task_score_mean"]
) / analysis_df["task_score_sd"]

analysis_df.loc[analysis_df["task_score_sd"].eq(0), "slogan_creativity_z"] = np.nan

analysis_csv_path = DIRS["compiled"] / f"slogans_with_gpt54_creativity_scores_z_IMPUTED__{RUN_ID}.csv"
analysis_pkl_path = DIRS["compiled"] / f"slogans_with_gpt54_creativity_scores_z_IMPUTED__{RUN_ID}.pkl"

analysis_df.to_csv(analysis_csv_path, index=False)
analysis_df.to_pickle(analysis_pkl_path)

print("Saved row-level z-scored slogan file:")
print(analysis_csv_path)
print(analysis_pkl_path)

display(
    analysis_df
    .groupby("task_id", observed=True)
    .agg(
        n=("slogan_creativity_1to5", "size"),
        mean_raw=("slogan_creativity_1to5", "mean"),
        sd_raw=("slogan_creativity_1to5", "std"),
        mean_z=("slogan_creativity_z", "mean"),
        sd_z=("slogan_creativity_z", "std"),
        n_imputed=("slogan_creativity_imputed", lambda x: int(x.fillna(False).astype(bool).sum())),
    )
    .reset_index()
)

Saved row-level z-scored slogan file:
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogans_with_gpt54_creativity_scores_z_IMPUTED__20260525_030656__bf5164cf.csv
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogans_with_gpt54_creativity_scores_z_IMPUTED__20260525_030656__bf5164cf.pkl


,task_id,n,mean_raw,sd_raw,mean_z,sd_z,n_imputed
0,slogan_blood_donation,5400,4.000926,0.661288,2.368887e-16,1.0,1
1,slogan_smartphone,5400,3.844630,0.505323,2.390063e-17,1.0,0
2,slogan_soda,5400,3.781481,0.542361,-3.352462e-16,1.0,0


In [184]:
# Cell D — Create compact analysis-ready slogan judge file

preferred_cols = [
    # keys / design
    "row_uid",
    "_source_row_pos",
    "source_stage",
    "provider",
    "model",
    "_provider_model_key",
    "task_id",
    "task_label",
    "task_family",
    "method",
    "method_analysis",
    "strategy",
    "condition",
    "round",
    "slot_id",
    "slot_num",
    "request_key",

    # text
    "slogan_text",

    # judge output
    "slogan_creativity_1to5",
    "slogan_creativity_z",
    "task_score_mean",
    "task_score_sd",
    "slogan_creativity_imputed",
    "imputation_note",

    # provenance
    "judge_request_key",
    "judge_status",
    "judge_raw_text",
    "provider_response_id",
    "batch_id",
]

compact_cols = [c for c in preferred_cols if c in analysis_df.columns]

compact_judge_df = analysis_df[compact_cols].copy()

compact_csv_path = DIRS["compiled"] / f"slogan_gpt54_judge_scores_ANALYSIS_READY__{RUN_ID}.csv"
compact_pkl_path = DIRS["compiled"] / f"slogan_gpt54_judge_scores_ANALYSIS_READY__{RUN_ID}.pkl"

compact_judge_df.to_csv(compact_csv_path, index=False)
compact_judge_df.to_pickle(compact_pkl_path)

print("Saved compact analysis-ready judge file:")
print(compact_csv_path)
print(compact_pkl_path)

print("\nShape:", compact_judge_df.shape)
print("Missing raw scores:", compact_judge_df["slogan_creativity_1to5"].isna().sum())
print("Missing z scores:", compact_judge_df["slogan_creativity_z"].isna().sum())

display(compact_judge_df.head())

Saved compact analysis-ready judge file:
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogan_gpt54_judge_scores_ANALYSIS_READY__20260525_030656__bf5164cf.csv
ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogan_gpt54_judge_scores_ANALYSIS_READY__20260525_030656__bf5164cf.pkl

Shape: (16200, 27)
Missing raw scores: 0
Missing z scores: 0


,row_uid,_source_row_pos,source_stage,provider,model,_provider_model_key,task_id,task_label,task_family,method,...,slogan_text,slogan_creativity_1to5,slogan_creativity_z,task_score_mean,task_score_sd,slogan_creativity_imputed,imputation_note,judge_request_key,judge_status,judge_raw_text
0,934d7d0e8b4fc18e9615f3b13d10e1cd,0,original,openai,gpt-5.4,openai :: gpt-5.4,slogan_smartphone,Smartphone slogan,slogan,indep,...,"Tomorrow, perfectly in your palm.",4.0,0.307467,3.84463,0.505323,False,,sloganjudge__f967b307f0861efc91a8fdd20cac9bc8,success,4
1,d1ea941deee73c472397cc3ca47a7437,1,original,openai,gpt-5.4,openai :: gpt-5.4,slogan_smartphone,Smartphone slogan,slogan,self,...,"Infinity, elegantly within reach.",4.0,0.307467,3.84463,0.505323,False,,sloganjudge__054c9c0ef01e16a3f62b82d80ca37a72,success,4
2,3b64acd2905951b075b64bb31104a930,2,original,openai,gpt-5.4,openai :: gpt-5.4,slogan_smartphone,Smartphone slogan,slogan,indep,...,"Tomorrow, perfectly in your palm.",4.0,0.307467,3.84463,0.505323,False,,sloganjudge__695c58007bf270423de45dde2bc1e3e4,success,4
3,4dd3fc0843cb5066d8b517ff73eb8db9,3,original,openai,gpt-5.4,openai :: gpt-5.4,slogan_smartphone,Smartphone slogan,slogan,self,...,Pocket the Possible.,4.0,0.307467,3.84463,0.505323,False,,sloganjudge__addb67461a913d296b4e8f11ee345efa,success,4
4,75cccdf982363ad722fc6fde7c0fedbb,4,original,openai,gpt-5.4,openai :: gpt-5.4,slogan_smartphone,Smartphone slogan,slogan,indep,...,"Pocket the Future, Effortlessly.",4.0,0.307467,3.84463,0.505323,False,,sloganjudge__43d47f528bccb01d150d6136edee7f0c,success,4


In [185]:
# Cell E — Copy analysis-ready judge files to final_analysis/input_data

import shutil

judge_input_dir = FINAL_INPUT_DIR / "slogan_llm_judge_gpt54"
judge_input_dir.mkdir(parents=True, exist_ok=True)

dest_compact_csv = judge_input_dir / "slogan_gpt54_judge_scores_ANALYSIS_READY.csv"
dest_compact_pkl = judge_input_dir / "slogan_gpt54_judge_scores_ANALYSIS_READY.pkl"

dest_summary_csv = judge_input_dir / "slogan_creativity_summary_by_cell.csv"
dest_summary_pkl = judge_input_dir / "slogan_creativity_summary_by_cell.pkl"

shutil.copy2(compact_csv_path, dest_compact_csv)
shutil.copy2(compact_pkl_path, dest_compact_pkl)
shutil.copy2(summary_csv_path, dest_summary_csv)
shutil.copy2(summary_pkl_path, dest_summary_pkl)

manifest = {
    "created_at_utc": now_iso(),
    "run_id": RUN_ID,
    "source_run_dir": str(DATA_ROOT),
    "row_level_source_csv": str(compact_csv_path),
    "row_level_source_pkl": str(compact_pkl_path),
    "summary_source_csv": str(summary_csv_path),
    "summary_source_pkl": str(summary_pkl_path),
    "dest_compact_csv": str(dest_compact_csv),
    "dest_compact_pkl": str(dest_compact_pkl),
    "dest_summary_csv": str(dest_summary_csv),
    "dest_summary_pkl": str(dest_summary_pkl),
    "n_rows": int(len(compact_judge_df)),
    "n_imputed": int(compact_judge_df.get("slogan_creativity_imputed", pd.Series(False, index=compact_judge_df.index)).fillna(False).astype(bool).sum()),
    "imputation_note": "One BatchAPI 500 execution failure was imputed with provider-model-task-method-strategy cell median.",
    "score_col": "slogan_creativity_1to5",
    "z_score_col": "slogan_creativity_z",
}

dest_manifest = judge_input_dir / "slogan_gpt54_judge_scores_manifest.json"
write_json(dest_manifest, manifest)

print("Copied analysis-ready files to:")
print(judge_input_dir)
print("\nManifest:")
print(dest_manifest)
print(json.dumps(manifest, indent=2))

Copied analysis-ready files to:
final_analysis/input_data/slogan_llm_judge_gpt54

Manifest:
final_analysis/input_data/slogan_llm_judge_gpt54/slogan_gpt54_judge_scores_manifest.json
{
  "created_at_utc": "2026-05-25T13:23:33.352445+00:00",
  "run_id": "20260525_030656__bf5164cf",
  "source_run_dir": "ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf",
  "row_level_source_csv": "ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogan_gpt54_judge_scores_ANALYSIS_READY__20260525_030656__bf5164cf.csv",
  "row_level_source_pkl": "ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5164cf/08_compiled/slogan_gpt54_judge_scores_ANALYSIS_READY__20260525_030656__bf5164cf.pkl",
  "summary_source_csv": "ai_data/deflect_creativity/judging/judge_gpt-5.4/slogan_llm_judge_gpt54_creativity/run_20260525_030656__bf5

In [188]:
# Final repair/export cell — write slogan judge quality rows in clean-analysis schema

# ================================================================
# Export slogan LLM-judge scores with the exact schema expected by
# the clean full-analysis notebook Cell 10.
# ================================================================

from pathlib import Path
import json
import datetime as dt
import numpy as np
import pandas as pd
import shutil

# ----------------------------------------------------------------
# Locate project input directory
# ----------------------------------------------------------------

if "FINAL_INPUT_DIR" not in globals():
    FINAL_INPUT_DIR = Path("final_analysis") / "input_data"

FINAL_INPUT_DIR = Path(FINAL_INPUT_DIR)
FINAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)

STAGED_DIR = FINAL_INPUT_DIR / "slogan_llm_judge_gpt54"
STAGED_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_DIR = FINAL_INPUT_DIR / "llm_judge_quality"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

FOLLOWUP_QUALITY_DIR = FINAL_INPUT_DIR / "followup_new_baselines" / "quality"
FOLLOWUP_QUALITY_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------
# Load the row-level judge data from the best available object/source
# ----------------------------------------------------------------

if "analysis_df" in globals():
    judge_df = analysis_df.copy()
    source_note = "analysis_df"
elif "compact_judge_df" in globals():
    judge_df = compact_judge_df.copy()
    source_note = "compact_judge_df"
elif "compact_pkl_path" in globals() and Path(compact_pkl_path).exists():
    judge_df = pd.read_pickle(compact_pkl_path)
    source_note = str(compact_pkl_path)
else:
    fallback_path = STAGED_DIR / "slogan_gpt54_judge_scores_ANALYSIS_READY.pkl"
    if fallback_path.exists():
        judge_df = pd.read_pickle(fallback_path)
        source_note = str(fallback_path)
    else:
        raise FileNotFoundError(
            "Could not find analysis_df, compact_judge_df, compact_pkl_path, "
            f"or fallback file: {fallback_path}"
        )

print("Loaded slogan judge rows from:", source_note)
print("Raw shape:", judge_df.shape)
print("Columns:", list(judge_df.columns))

# ----------------------------------------------------------------
# Required columns and method/strategy normalization
# ----------------------------------------------------------------

score_col = "slogan_creativity_1to5"

required_base_cols = [
    "provider",
    "task_id",
    "task_family",
    "method",
    "strategy",
    "row_uid",
    score_col,
]

missing = [c for c in required_base_cols if c not in judge_df.columns]
if missing:
    raise RuntimeError(f"Judge data missing required base columns: {missing}")

judge_df[score_col] = pd.to_numeric(judge_df[score_col], errors="coerce")

if judge_df[score_col].isna().any():
    display(judge_df[judge_df[score_col].isna()].head(100))
    raise RuntimeError("Some slogan judge rows still have missing creativity scores.")

# The judge notebook uses:
#   method_analysis = clean full-analysis method ID
#   method          = paper/judge shorthand
# If method_analysis exists, use it as the clean-analysis method.
judge_method_to_analysis = {
    "indep": "one_shot",
    "self": "self_rewrite_e0",
    "peer1": "self_plus_1_peer",
    "peer2": "self_plus_2_peers",
    "strat": "simplestrat5",
    "repr": "g2_css_static3",
    "one_shot": "one_shot",
    "self_rewrite_e0": "self_rewrite_e0",
    "self_plus_1_peer": "self_plus_1_peer",
    "self_plus_2_peers": "self_plus_2_peers",
    "simplestrat5": "simplestrat5",
    "g2_css_static3": "g2_css_static3",
}

analysis_to_paper_method = {
    "one_shot": "indep",
    "self_rewrite_e0": "self",
    "self_plus_1_peer": "peer1",
    "self_plus_2_peers": "peer2",
    "simplestrat5": "strat",
    "g2_css_static3": "repr",
}

if "method_analysis" in judge_df.columns:
    judge_df["method_paper"] = judge_df["method"].astype(str)
    judge_df["method"] = judge_df["method_analysis"].astype(str)
else:
    judge_df["method_paper"] = judge_df["method"].astype(str)
    judge_df["method"] = judge_df["method"].astype(str).map(judge_method_to_analysis)

bad_method_rows = judge_df[judge_df["method"].isna()].copy()
if not bad_method_rows.empty:
    display(bad_method_rows[["provider", "task_id", "method_paper", "strategy"]].head(100))
    raise RuntimeError("Could not map some judge method labels to clean-analysis method IDs.")

judge_df["method_paper"] = judge_df["method"].map(analysis_to_paper_method).fillna(judge_df["method_paper"])

def normalize_strategy_for_clean(x):
    s = str(x).strip().lower()
    if s in {"neutral", "vanilla", "base", "control"}:
        return "vanilla"
    if s in {"diverge", "divergent", "diversity", "diverse"}:
        return "diverge"
    if "diverg" in s or "divers" in s:
        return "diverge"
    return s

judge_df["strategy"] = judge_df["strategy"].map(normalize_strategy_for_clean)

# ----------------------------------------------------------------
# Add clean-analysis quality columns
# ----------------------------------------------------------------

judge_df["quality_raw"] = judge_df[score_col]
judge_df["quality_score_raw"] = judge_df["quality_raw"]
judge_df["quality_value"] = judge_df["quality_raw"]
judge_df["quality_metric"] = "slogan_llm_judge_gpt54_creativity_1to5"
judge_df["quality_source"] = "slogan_llm_judge_gpt54"

if "slogan_creativity_z" in judge_df.columns:
    judge_df["quality_z_task_precomputed_ddof1"] = judge_df["slogan_creativity_z"]

# Optional columns used for provenance. Create if missing.
for c in [
    "model",
    "task_label",
    "source_stage",
    "request_key",
    "slot_id",
    "slogan_text",
    "judge_request_key",
    "judge_status",
    "judge_raw_text",
    "provider_response_id",
    "batch_id",
    "slogan_creativity_imputed",
    "imputation_note",
]:
    if c not in judge_df.columns:
        judge_df[c] = np.nan

# Mark imputed rows as usable if they have a score.
if "slogan_creativity_imputed" in judge_df.columns:
    judge_df["slogan_creativity_imputed"] = (
        judge_df["slogan_creativity_imputed"].fillna(False).astype(bool)
    )

# ----------------------------------------------------------------
# Build final export table
# ----------------------------------------------------------------

export_cols = [
    "provider",
    "model",
    "task_family",
    "task_id",
    "task_label",
    "method",
    "method_paper",
    "strategy",
    "source_stage",
    "row_uid",
    "request_key",
    "slot_id",
    "slogan_text",
    "judge_request_key",
    "judge_status",
    "judge_raw_text",
    "provider_response_id",
    "batch_id",
    "slogan_creativity_imputed",
    "imputation_note",
    "quality_raw",
    "quality_score_raw",
    "quality_value",
    "quality_metric",
    "quality_source",
    "slogan_creativity_1to5",
    "quality_z_task_precomputed_ddof1",
]

export_cols = [c for c in export_cols if c in judge_df.columns]

judge_quality_rows = judge_df[export_cols].copy()

# ----------------------------------------------------------------
# Coverage checks
# ----------------------------------------------------------------

expected_providers = ["openai", "anthropic", "gemini"]
expected_slogan_tasks = [
    "slogan_smartphone",
    "slogan_soda",
    "slogan_blood_donation",
]
expected_methods = [
    "one_shot",
    "self_rewrite_e0",
    "self_plus_1_peer",
    "self_plus_2_peers",
    "simplestrat5",
    "g2_css_static3",
]
expected_strategies = ["vanilla", "diverge"]
expected_n_per_cell = 150

expected_rows = (
    len(expected_providers)
    * len(expected_slogan_tasks)
    * len(expected_methods)
    * len(expected_strategies)
    * expected_n_per_cell
)

if len(judge_quality_rows) != expected_rows:
    display(
        judge_quality_rows
        .groupby(["provider", "task_id", "method", "strategy"], observed=True)
        .size()
        .reset_index(name="n")
        .sort_values(["provider", "task_id", "method", "strategy"])
    )
    raise RuntimeError(
        f"Expected {expected_rows:,} row-level slogan judge rows, got {len(judge_quality_rows):,}."
    )

coverage = (
    judge_quality_rows
    .groupby(["provider", "task_id", "method", "strategy"], observed=True)
    .agg(
        n=("quality_raw", "size"),
        n_scored=("quality_raw", lambda x: x.notna().sum()),
        mean_quality_raw=("quality_raw", "mean"),
        sd_quality_raw=("quality_raw", "std"),
    )
    .reset_index()
    .sort_values(["provider", "task_id", "method", "strategy"])
)

bad_coverage = coverage[
    coverage["n"].ne(expected_n_per_cell)
    | coverage["n_scored"].ne(expected_n_per_cell)
].copy()

if not bad_coverage.empty:
    display(bad_coverage)
    raise RuntimeError("Coverage check failed: some cells are not 150/150 scored.")

if judge_quality_rows["row_uid"].duplicated().any():
    dupes = judge_quality_rows[judge_quality_rows["row_uid"].duplicated(keep=False)]
    display(dupes.head(100))
    raise RuntimeError("Duplicate row_uid values in judge quality export.")

# ----------------------------------------------------------------
# Write stable files expected by clean analysis notebook
# ----------------------------------------------------------------

latest_pkl_path = FINAL_INPUT_DIR / "slogan_llm_judge_quality_rows_analysis_ready.pkl"
latest_csv_path = FINAL_INPUT_DIR / "slogan_llm_judge_quality_rows_analysis_ready.csv"
latest_coverage_path = FINAL_INPUT_DIR / "slogan_llm_judge_quality_rows_coverage_audit.csv"

run_id = globals().get("RUN_ID", dt.datetime.now().strftime("%Y%m%d_%H%M%S") + "__manual")

archive_pkl_path = ARCHIVE_DIR / f"slogan_llm_judge_quality_rows_analysis_ready__{run_id}.pkl"
archive_csv_path = ARCHIVE_DIR / f"slogan_llm_judge_quality_rows_analysis_ready__{run_id}.csv"
archive_coverage_path = ARCHIVE_DIR / f"slogan_llm_judge_quality_rows_coverage_audit__{run_id}.csv"

followup_copy_pkl_path = FOLLOWUP_QUALITY_DIR / "slogan_llm_judge_quality_rows_analysis_ready.pkl"
followup_copy_csv_path = FOLLOWUP_QUALITY_DIR / "slogan_llm_judge_quality_rows_analysis_ready.csv"

judge_quality_rows.to_pickle(latest_pkl_path)
judge_quality_rows.to_csv(latest_csv_path, index=False)
coverage.to_csv(latest_coverage_path, index=False)

judge_quality_rows.to_pickle(archive_pkl_path)
judge_quality_rows.to_csv(archive_csv_path, index=False)
coverage.to_csv(archive_coverage_path, index=False)

judge_quality_rows.to_pickle(followup_copy_pkl_path)
judge_quality_rows.to_csv(followup_copy_csv_path, index=False)

manifest = {
    "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "run_id": run_id,
    "source_note": source_note,
    "quality_metric": "slogan_llm_judge_gpt54_creativity_1to5",
    "quality_source": "slogan_llm_judge_gpt54",
    "score_col": "slogan_creativity_1to5",
    "quality_raw_col": "quality_raw",
    "expected_rows": int(expected_rows),
    "observed_rows": int(len(judge_quality_rows)),
    "n_imputed": int(
        judge_quality_rows.get(
            "slogan_creativity_imputed",
            pd.Series(False, index=judge_quality_rows.index),
        )
        .fillna(False)
        .astype(bool)
        .sum()
    ),
    "latest_pkl_path": str(latest_pkl_path),
    "latest_csv_path": str(latest_csv_path),
    "latest_coverage_path": str(latest_coverage_path),
    "archive_pkl_path": str(archive_pkl_path),
    "archive_csv_path": str(archive_csv_path),
    "followup_copy_pkl_path": str(followup_copy_pkl_path),
    "note_for_clean_analysis": (
        "This file contains row-level slogan LLM-judge quality scores. "
        "Clean analysis Cell 10 should load quality_raw and recompute task-standardized quality_z_task."
    ),
}

manifest_path = FINAL_INPUT_DIR / "slogan_llm_judge_quality_rows_manifest_latest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Exported clean-analysis-ready slogan judge quality rows.")
print("Latest PKL:", latest_pkl_path)
print("Latest CSV:", latest_csv_path)
print("Coverage audit:", latest_coverage_path)
print("Archive PKL:", archive_pkl_path)
print("Follow-up copy:", followup_copy_pkl_path)
print("Manifest:", manifest_path)

display(coverage.head(30))

display(
    judge_quality_rows
    .groupby(["provider", "method", "strategy"], observed=True)
    .agg(
        n=("quality_raw", "size"),
        mean_quality_raw=("quality_raw", "mean"),
        sd_quality_raw=("quality_raw", "std"),
        n_imputed=("slogan_creativity_imputed", lambda x: int(x.fillna(False).astype(bool).sum()))
        if "slogan_creativity_imputed" in judge_quality_rows.columns
        else ("quality_raw", "size"),
    )
    .reset_index()
    .sort_values(["provider", "method", "strategy"])
)

Loaded slogan judge rows from: analysis_df
Raw shape: (16200, 101)
Columns: ['_old_row_pos', 'agent_id', 'agent_index', 'anchor_count', 'batch_custom_id', 'batch_id_x', 'batch_name', 'batch_output_file', 'condition', 'context_count', 'created_at_utc', 'error', 'experiment_id', 'finish_reason', 'group_id', 'idea_region', 'include_thinking_config_in_batch', 'is_success', 'max_output_tokens', 'method', 'method_analysis', 'method_label', 'model', 'original_source_mtime', 'original_source_path', 'parsed_at_utc', 'peer_round1_agent_ids', 'peer_round1_texts_json', 'provider', 'provider_label', 'provider_response_id_x', 'raw_record', 'raw_result_type', 'request_key', 'rough_sentence_count', 'round', 'row_uid', 'self_round1_request_key', 'self_round1_text', 'slogan_word_violation', 'slot_id', 'slot_num', 'source_long_path', 'source_mtime', 'source_run_dir', 'source_stage', 'staged_source_path', 'status', 'stop_reason', 'story_sentence_flag', 'strategy', 'stratum_id', 'system_instructions', 'tas

,provider,task_id,method,strategy,n,n_scored,mean_quality_raw,sd_quality_raw
0,anthropic,slogan_blood_donation,g2_css_static3,diverge,150,150,4.213333,0.574476
1,anthropic,slogan_blood_donation,g2_css_static3,vanilla,150,150,4.146667,0.483092
2,anthropic,slogan_blood_donation,one_shot,diverge,150,150,4.340000,0.541313
3,anthropic,slogan_blood_donation,one_shot,vanilla,150,150,4.386667,0.540776
4,anthropic,slogan_blood_donation,self_plus_1_peer,diverge,150,150,4.140000,0.555992
5,anthropic,slogan_blood_donation,self_plus_1_peer,vanilla,150,150,4.126667,0.508936
6,anthropic,slogan_blood_donation,self_plus_2_peers,diverge,150,150,4.340000,0.553573
7,anthropic,slogan_blood_donation,self_plus_2_peers,vanilla,150,150,4.273333,0.664504
8,anthropic,slogan_blood_donation,self_rewrite_e0,diverge,150,150,4.026667,0.601415
9,anthropic,slogan_blood_donation,self_rewrite_e0,vanilla,150,150,4.253333,0.494081


,provider,method,strategy,n,mean_quality_raw,sd_quality_raw,n_imputed
0,anthropic,g2_css_static3,diverge,450,3.966667,0.500556,0
1,anthropic,g2_css_static3,vanilla,450,3.904444,0.501426,0
2,anthropic,one_shot,diverge,450,4.177778,0.494463,0
3,anthropic,one_shot,vanilla,450,4.082222,0.512555,0
4,anthropic,self_plus_1_peer,diverge,450,4.017778,0.442351,0
5,anthropic,self_plus_1_peer,vanilla,450,3.980000,0.444767,0
6,anthropic,self_plus_2_peers,diverge,450,4.082222,0.537995,0
7,anthropic,self_plus_2_peers,vanilla,450,4.031111,0.516033,0
8,anthropic,self_rewrite_e0,diverge,450,4.004444,0.503863,0
9,anthropic,self_rewrite_e0,vanilla,450,4.057778,0.433810,0
